# Laboratori 8 — Floyd-Warshall & Rrjedhat Maksimale: Edmonds-Karp


---

## 1 — Floyd-Warshall: Rrugët më të Shkurtra Ndërmjet Të Gjitha Çifteve

### Çfarë problemi zgjidhim?

Imagjinoni një **rrjet transporti urban** ku:
- Kemi **depo**, **qendra rajonale** dhe **pika dorëzimi** të lidhura me rrugë
- Çdo rrugë ka **kohë udhëtimi** të ndryshme (trafik, distancë, lloj rruge)
- Duam të dimë **rrugën minimale** ndërmjet **çdo çifti** vendesh — jo vetëm nga një pikë

### Pse kjo është ndryshe nga Dijkstra?

| Aspekti | Dijkstra | Floyd-Warshall |
|---------|----------|----------------|
| **Burimet** | Një burim | Të gjithë çiftet |
| **Qasja** | Greedy (zgjedhje lokale) | Programim dinamik |
| **Output** | d[s][v] për çdo v | D[i][j] për çdo (i,j) |
| **Kompleksiteti** | O(E log V) për një burim | O(V³) — të gjitha çiftet |
| **Kur preferohet** | Burim i vetëm | Pyetje të shumta, grafe të vogla |

### Ideja e Programimit Dinamik

**Pyetja themelore:** A kalon rruga optimale nga `i` tek `j` nëpër kulmin `k`?

```
D[i][j][k] = min(
    D[i][j][k-1],                    ← rruga nuk kalon nëpër k
    D[i][k][k-1] + D[k][j][k-1]     ← rruga kalon nëpër k si ndërmjetës
)
```

Pas optimizimit të hapësirës (duke mbajtur vetëm matricën e fundit):

```
D[i][j] = min(D[i][j],  D[i][k] + D[k][j])   ← provoj k si ndërmjetës
```

### Pseudokodi i Algoritmit

```
FloydWarshall(W, n):
    D ← W                      # inicializo me peshat e harqeve
    P[i][j] ← i  nëse ekziston harku (i,j), tjetër None

    për k = 0 deri n-1:        ← çdo kulm si ndërmjetës i mundshëm
        për i = 0 deri n-1:
            për j = 0 deri n-1:
                nëse D[i][k] + D[k][j] < D[i][j]:
                    D[i][j] ← D[i][k] + D[k][j]   # gjeta rrugë më të shkurtër!
                    P[i][j] ← P[k][j]              # regjistro ndërmjetësin

    nëse D[v][v] < 0 për ndonjë v: RAPORTIM GABIM (cikël negativ)
    kthe D, P
```

### Dy Matricat e Daljes

| Matrica | Kuptimi | Shembull |
|---------|---------|---------|
| **D[i][j]** | Distanca minimale nga i tek j | D[DQ][PD-4] = 41 min |
| **P[i][j]** | Paraardhësi i j në rrugën optimale nga i | P[DQ][PD-4] = QR-C |

### Rindërtimi i Rrugës nga P[i][j]

```python
def reconstruct(P, src, dst):
    path = [dst]
    while path[-1] != src:
        path.append(P[src][path[-1]])
    path.reverse()
    return path   # src → ... → dst
```

### Trajtimi i Cikleve Negative

Nëse pas ekzekutimit `D[v][v] < 0` për ndonjë kulm `v`, grafi ka **cikël negativ** —
rruga mund të bëhet pafundësisht e shkurtër duke kaluar vazhdimisht nëpër të.
Algoritmi e detekton këtë dhe duhet të raportojë gabim.


---

## Ushtrimi 1 — Sistemi Logjistik i Shpërndarjes Urbane (UrbanPost)

-----------------------------------------------------------------------------

### Skenari Real

Kompania **UrbanPost** menaxhon shpërndarjen e paketave në një qytet.
Ata kanë **depo qendrore**, **qendra ndërmjetëse rajonale** dhe **pika dorëzimi**,
të lidhura me rrugë që kanë kohë udhëtimi të ndryshme (minuta) — duke marrë
parasysh trafikun, distancën dhe llojin e rrugës.

Çdo ditë, kompania duhet t'u përgjigjet pyetjeve të tilla:

- *"Cila është rruga më e shpejtë nga Depoja Qendrore deri te Pika X?"*
- *"Sa kohë nevojitet minimalisht për të lidhur çdo qendër rajonale me çdo pikë dorëzimi?"*
- *"Nëse bllokohet një rrugë, cilat lidhje alternative ekzistojnë?"*

Meqë pyetjet janë të shumta dhe të vazhdueshme gjatë ditës, kompania llogarit
**paraprakisht** të gjitha distancat minimale midis çdo çifti pikash — kjo është
pikërisht ajo çfarë ofron Floyd-Warshall.

-----------------------------------------------------------------------------

### Pse Floyd-Warshall dhe jo Dijkstra?

#### Në rrjete logjistike reale:

- Kamioni mund të nisë nga **çdo pikë** — jo vetëm nga depoja qendrore
- Pyetjet vijnë nga **çdo drejtuesi** në çdo çast të ditës
- Duhet **koha e llogaritur paraprakisht**, jo rillogaritje gjatë punës
- Grafi ka vetëm **8 nyje** — O(8³) = 512 operacione, jashtëzakonisht i shpejtë

#### Algoritmi Floyd-Warshall:

- llogarit **të gjitha çiftet** me një ekzekutim të vetëm
- prodhon **matricën e plotë** D[i][j] — gati për çdo pyetje
- rindërton **rrugën e plotë** nëpërmjet matricës P[i][j]
- detekton **cikle negative** automatikisht

-----------------------------------------------------------------------------

### Struktura e Rrjetit

```
DEPO QENDRORE ──→ [QR-A, QR-B, QR-C] ──→ [PD-1, PD-2, PD-3, PD-4]
   (DQ)             (Qendra Rajonale)        (Pika Dorëzimi)
```

**Detajet e grafit:**
- **8 kulme:** DQ (1) + Qendra Rajonale (3) + Pika Dorëzimi (4)
- **21 harqe të drejtuara** me kohë udhëtimi 6–24 minuta
- **Graf i drejtuar:** disa rrugë janë njëkaheshe (trafik me sens të vetëm)
- **Pa cikle negative:** peshat janë kohë reale udhëtimi > 0

### Çfarë Simulojmë?

Animimi tregon hap pas hapi si Floyd-Warshall **provon çdo kulm `k`** si ndërmjetës
të mundshëm për të përmirësuar rrugën D[i][j]:

| Ngjyra në Animim | Kuptimi |
|------------------|---------|
| 🟡 **E verdhë (k)** | Kulmi ndërmjetës aktual që po provohet |
| 🔵 **Cyan** | Harqet i→k dhe k→j që po kontrollohen |
| 🟢 **E gjelbër** | Qeliza D[i][j] e sapo përmirësuar |
| 🟣 **Vjollcë** | Rruga e kërkuar ndërmjet dy pikave specifike |

### Si ta Arrijmë?

1. **Inicializimi:** D[i][j] = pesha e harkut (i,j) nëse ekziston, ∞ ndryshe, 0 në diagonal
2. **Tre laqe:** `for k → for i → for j: D[i][j] = min(D[i][j], D[i][k]+D[k][j])`
3. **Snapshot:** çdo përditësim i D[i][j] ruhet si hap animimi
4. **Rindërtim:** P[i][j] ruan ndërmjetësin — ndiqet pas→para deri te burimi

**Tastet gjatë simulimit:**
- `SPACE` — luaj/pauzo | `S` — hap i vetëm | `R` — rinicio
- `Q/W` — ndrysho burimin e pyetjes | `E/D` — ndrysho destinacionin
- `G` — shfaq/fshih rrugën e kërkuar | `F11` — ekran i plotë | `ESC` — dil


### Implementimi i Ushtrimit 1 — UrbanPost


In [3]:
"""
Ushtrimi 1 — Floyd-Warshall: Sistemi Logjistik i Shpërndarjes Urbane
Kompania UrbanPost — Depo Qendrore, Qendra Rajonale, Pika Dorëzimi
Animim hap-pas-hapi i algoritmit me vizualizim tematik

NDRYSHIME:
- Kamionët lëvizin VETËM mbi brinjët aktive (i→k dhe k→j)
- Shigjetat me ngjyrë të dalluar sipas fazës dhe madhësi më e madhe
- Algoritmi iterOn GJITHMONË për çdo (i,j) — edhe kur vlera është INF
  (faza "check_inf" tregon se jemi duke kontrolluar por s'ka rrugë të mundshme)
"""

import pygame, sys, math
from copy import deepcopy

pygame.init()

# ─────────────────────── PALETA E NGJYRAVE ────────────────────────
C = {
    "bg":        (6, 10, 22),
    "graph":     (9, 15, 32),
    "panel":     (12, 20, 42),
    "panel2":    (17, 28, 55),
    "white":     (230, 245, 255),
    "dim":       (110, 135, 180),
    "blue":      (65, 185, 255),
    "cyan":      (35, 215, 250),
    "yellow":    (245, 192, 38),
    "red":       (255, 75, 55),
    "green":     (55, 215, 115),
    "orange":    (255, 155, 55),
    "purple":    (160, 120, 255),
    "road":      (55, 72, 95),
    "road_hi":   (95, 125, 165),
    "depo":      (190, 140, 55),
    "qr":        (55, 140, 190),
    "pd":        (90, 175, 90),
    "active_k":  (255, 210, 50),
    "inactive":  (45, 58, 82),
    # Ngjyrat e shigjetave — të dalluara qartë
    "arrow_default": (140, 170, 210),   # gri-blu
    "arrow_update":  (55, 215, 115),    # jeshil i ndezur
    "arrow_check":   (245, 192, 38),    # verdhë
    "arrow_inf":     (80, 95, 120),     # gri i errët — s'ka rrugë
    "arrow_ik":      (35, 215, 250),    # cyan — segmenti i→k
    "arrow_kj":      (255, 155, 55),    # portokalli — segmenti k→j
}

INF = float('inf')
FONTS = {}

# ─────────────────────── GRAFIKU I RRJETIT ────────────────────────
NODE_NAMES = [
    "Depo\nQendrore",   # 0  DQ
    "QR-A",             # 1
    "QR-B",             # 2
    "QR-C",             # 3
    "PD-1",             # 4
    "PD-2",             # 5
    "PD-3",             # 6
    "PD-4",             # 7
]
SHORT = ["0", "1", "2", "3", "4", "5", "6", "7"]
N = len(NODE_NAMES)

# (from, to, weight_minutes) — graf i drejtuar
EDGES_RAW = [
    (0, 1, 12), (0, 2, 18), (0, 3, 22),
    (1, 2,  7), (1, 4, 15), (1, 5, 20),
    (2, 1,  7), (2, 3,  9), (2, 5, 11), (2, 6, 17),
    (3, 2,  9), (3, 6, 13), (3, 7, 19),
    (4, 5,  6), (4, 7, 24),
    (5, 4,  6), (5, 6,  8), (5, 7, 14),
    (6, 5,  8), (6, 7, 10),
    (7, 6, 10),
]

# Set i shpejtë për kërkim
EDGE_SET = {(u, v) for u, v, _ in EDGES_RAW}

# ─────────────────────── FONT HELPER ──────────────────────────────
def fnt(size, bold=False):
    k = (size, bold)
    if k not in FONTS:
        FONTS[k] = pygame.font.SysFont("Segoe UI", size, bold=bold)
    return FONTS[k]

def txt(s, msg, size, col, pos, center=False, bold=False):
    img = fnt(size, bold).render(str(msg), True, col)
    r = img.get_rect(center=pos) if center else img.get_rect(topleft=pos)
    s.blit(img, r)
    return r

def glow(s, x, y, col, r):
    for rr in range(r, 0, -14):
        a = int(70 * rr / r)
        g = pygame.Surface((rr * 2, rr * 2), pygame.SRCALPHA)
        pygame.draw.circle(g, (*col, a), (rr, rr), rr)
        s.blit(g, (x - rr, y - rr))

# ─────────────────────── FLOYD-WARSHALL ───────────────────────────
def build_matrix():
    D = [[INF] * N for _ in range(N)]
    P = [[None] * N for _ in range(N)]
    for i in range(N):
        D[i][i] = 0
    for u, v, w in EDGES_RAW:
        D[u][v] = w
        P[u][v] = u
    return D, P

def floyd_warshall_steps():
    """
    Gjeneroni të gjithë hapat e algoritmit me snapshot.
    RËNDËSISHME: Iterojmë GJITHMONË për çdo (i, j), edhe kur D[i][k] ose D[k][j] është INF.
    Kjo lejon vizualizimin të tregojë saktësisht ku ndodhemi në algoritëm.
    """
    D, P = build_matrix()
    steps = []
    steps.append(("init", deepcopy(D), deepcopy(P), -1, -1, -1, None))

    for k in range(N):
        for i in range(N):
            for j in range(N):
                if D[i][k] < INF and D[k][j] < INF:
                    new_val = D[i][k] + D[k][j]
                    if new_val < D[i][j]:
                        old = D[i][j]
                        D[i][j] = new_val
                        P[i][j] = P[k][j]
                        steps.append(("update", deepcopy(D), deepcopy(P),
                                      k, i, j, (old, new_val)))
                    else:
                        # Ka rrugë por nuk ka përmirësim
                        steps.append(("check", deepcopy(D), deepcopy(P),
                                      k, i, j, None))
                else:
                    # Një ose të dyja segmentet janë INF — s'ka rrugë nëpër k
                    steps.append(("check_inf", deepcopy(D), deepcopy(P),
                                  k, i, j, None))

        neg_cycle = any(D[v][v] < 0 for v in range(N))
        steps.append(("k_done", deepcopy(D), deepcopy(P), k, -1, -1, neg_cycle))

    steps.append(("done", deepcopy(D), deepcopy(P), -1, -1, -1, None))
    return steps, D, P

def reconstruct(P, src, dst):
    if P[src][dst] is None:
        return None
    path = [dst]
    while path[-1] != src:
        path.append(P[src][path[-1]])
    path.reverse()
    return path

# ─────────────────────── VIZATIMET E NYJEVE ───────────────────────
def draw_depo(s, x, y, active, tick):
    """
    Depo Qendrore — ndërtesë industriale.
    Kamionët lëvizin vetëm mbi brinjët aktive (menaxhohet nga draw_road).
    """
    if active:
        glow(s, x, y, C["depo"], 85)
    pygame.draw.rect(s, (140, 100, 38), (x - 42, y - 38, 84, 64), border_radius=4)
    pygame.draw.rect(s, (210, 165, 75), (x - 42, y - 38, 84, 64), 2, border_radius=4)
    pygame.draw.polygon(s, (165, 120, 48),
                        [(x - 48, y - 38), (x, y - 62), (x + 48, y - 38)])
    pygame.draw.polygon(s, (220, 180, 90),
                        [(x - 48, y - 38), (x, y - 62), (x + 48, y - 38)], 2)
    pygame.draw.rect(s, (55, 40, 18), (x - 20, y - 12, 40, 38))
    pygame.draw.rect(s, (180, 140, 60), (x - 20, y - 12, 40, 38), 2)
    for ox in (-28, 18):
        pygame.draw.rect(s, (100, 160, 220), (x + ox, y - 30, 16, 12))
        pygame.draw.rect(s, (180, 215, 250), (x + ox, y - 30, 16, 12), 1)

def draw_qr(s, x, y, label, active, tick):
    """Qendra Rajonale — kamioni lëviz VETËM kur aktive."""
    if active:
        glow(s, x, y, C["qr"], 72)
    pygame.draw.rect(s, (38, 98, 140), (x - 36, y - 30, 72, 54), border_radius=4)
    pygame.draw.rect(s, (90, 160, 210), (x - 36, y - 30, 72, 54), 2, border_radius=4)
    pygame.draw.rect(s, (50, 115, 160), (x - 40, y - 36, 80, 10))
    pygame.draw.rect(s, (30, 50, 70), (x - 12, y - 46, 24, 12), border_radius=3)
    pygame.draw.rect(s, (20, 60, 90), (x - 24, y - 20, 48, 36))
    txt(s, label, 18, C["cyan"], (x, y - 4), center=True, bold=True)

    if active:
        t = (tick * 0.009 + hash(label) * 0.4) % 1
        kx = int(x - 52 + t * 104)
        ky = y + 32
        pygame.draw.rect(s, (200, 200, 55), (kx - 14, ky - 7, 26, 11), border_radius=2)
        pygame.draw.rect(s, (60, 60, 60), (kx + 12, ky - 5, 8, 9), border_radius=2)
        for wx in (kx - 10, kx + 4):
            pygame.draw.circle(s, (25, 25, 25), (wx, ky + 4), 4)
            pygame.draw.circle(s, (100, 100, 100), (wx, ky + 4), 2)

def draw_pd(s, x, y, label, active):
    """Pikë Dorëzimi — shtëpi me kuti."""
    if active:
        glow(s, x, y, C["pd"], 65)
    pygame.draw.rect(s, (55, 108, 60), (x - 28, y - 22, 56, 44), border_radius=3)
    pygame.draw.rect(s, (100, 175, 105), (x - 28, y - 22, 56, 44), 2, border_radius=3)
    pygame.draw.polygon(s, (75, 140, 80),
                        [(x - 34, y - 22), (x, y - 44), (x + 34, y - 22)])
    pygame.draw.polygon(s, (130, 195, 135),
                        [(x - 34, y - 22), (x, y - 44), (x + 34, y - 22)], 2)
    pygame.draw.rect(s, (35, 70, 38), (x - 9, y - 4, 18, 26))
    pygame.draw.circle(s, (200, 170, 50), (x + 6, y + 8), 3)
    pygame.draw.rect(s, C["orange"], (x - 14, y + 22, 28, 18), border_radius=2)
    pygame.draw.line(s, (180, 120, 30), (x - 14, y + 31), (x + 14, y + 31), 2)
    pygame.draw.line(s, (180, 120, 30), (x, y + 22), (x, y + 40), 2)
    txt(s, label, 15, C["white"], (x, y + 52), center=True, bold=True)

# ─────────────────────── RRUGA / HARKU ─────────────────────────────
def draw_arrow(surf, p1, p2, color, width, sz=14):
    """
    Vizaton një shigjetë të dalluar qartë nga p1 → p2.
    Trekëndëshi i shigjetes është më i madh dhe ngjyrë e ndezur.
    """
    x1, y1 = p1; x2, y2 = p2
    dx, dy = x2 - x1, y2 - y1
    L = math.hypot(dx, dy) or 1
    ux, uy = dx / L, dy / L  # drejtim unit

    # Vendos shigjeten pak para fundit (70% e rrugës)
    tx = x1 + dx * 0.70
    ty = y1 + dy * 0.70

    a = math.atan2(dy, dx)
    tip   = (int(tx + ux * sz * 0.6), int(ty + uy * sz * 0.6))
    left  = (int(tx - sz * math.cos(a - 0.55)), int(ty - sz * math.sin(a - 0.55)))
    right = (int(tx - sz * math.cos(a + 0.55)), int(ty - sz * math.sin(a + 0.55)))

    # Kontur i errët për kontrast
    pygame.draw.polygon(surf, (0, 0, 0), [tip, left, right])
    # Trekëndëshi kryesor
    inner_tip   = (int(tx + ux * sz * 0.5), int(ty + uy * sz * 0.5))
    inner_left  = (int(tx - (sz-3) * math.cos(a - 0.5)), int(ty - (sz-3) * math.sin(a - 0.5)))
    inner_right = (int(tx - (sz-3) * math.cos(a + 0.5)), int(ty - (sz-3) * math.sin(a + 0.5)))
    pygame.draw.polygon(surf, color, [inner_tip, inner_left, inner_right])

def bezier_points(p1, p2, ctrl, steps=30):
    """Pikat e kurbës Bezier kuadratike nga p1 ne p2 me pike kontrolli ctrl."""
    pts = []
    for t in [i/steps for i in range(steps+1)]:
        bx = (1-t)**2*p1[0] + 2*(1-t)*t*ctrl[0] + t**2*p2[0]
        by = (1-t)**2*p1[1] + 2*(1-t)*t*ctrl[1] + t**2*p2[1]
        pts.append((int(bx), int(by)))
    return pts

def get_ctrl_point(p1, p2, u, v):
    """
    Kurbeza vetem brinjetet qe kapercejne nyje ne te njejten kolone (skip >= 2).
    Brinjetet fqinje (skip=1) vizatohen drejtvizore — jane te qarta.
    Brinjetet e kthimit (p.sh. 6->5) gjithashtu behen te kthyera pak.
    """
    x1,y1 = p1; x2,y2 = p2
    mx,my = (x1+x2)/2, (y1+y2)/2

    # Same-column PD (4,5,6,7)
    pd_order = [4, 5, 6, 7]  # renditja vertikale
    qr_order = [1, 2, 3]

    if u in pd_order and v in pd_order:
        ui = pd_order.index(u)
        vi = pd_order.index(v)
        skip = abs(vi - ui)
        if skip >= 2:
            # Kapercen nyje — kurbe e dukshme djathtas
            offset = 50 * skip
            return (mx + offset, my), True
        elif vi < ui:
            # Kthim prapa (p.sh. 5->4, 6->5, 7->6) — kurbe e vogel majtas
            return (mx - 38, my), True
        else:
            # Fqinj normal (4->5, 5->6, 6->7) — drejtvizore
            return (mx, my), False

    if u in qr_order and v in qr_order:
        ui = qr_order.index(u)
        vi = qr_order.index(v)
        skip = abs(vi - ui)
        if skip >= 2:
            offset = 45 * skip
            return (mx - offset, my), True
        elif vi < ui:
            # Kthim prapa (p.sh. 2->1, 3->2) — kurbe e vogel djathtas
            return (mx + 35, my), True
        else:
            return (mx, my), False

    # Brinje normale nderkolumonare — drejtvizore
    return (mx, my), False

def draw_road(s, p1, p2, line_color, arrow_color, width, tick,
              flowing=False, draw_truck=True, u=-1, v=-1):
    """
    Vizaton rrugën si kurbë Bezier (per brinjët e se njëjtës kolonë)
    ose drejtvizore (brinjët normale).
    """
    x1, y1 = p1; x2, y2 = p2
    dx, dy = x2 - x1, y2 - y1

    ctrl, curved = get_ctrl_point(p1, p2, u, v)

    if curved:
        pts = bezier_points(p1, p2, ctrl, steps=32)
        # Sfond (shadow)
        if len(pts) > 1:
            pygame.draw.lines(s, (12, 18, 32), False, pts, width + 6)
            pygame.draw.lines(s, line_color, False, pts, width)
            pygame.draw.lines(s, (200, 220, 240), False, pts, max(1, width // 6))

        # Kamion animuar mbi kurbë
        if flowing and draw_truck:
            t_idx = int((tick * 0.008 % 1) * (len(pts)-1))
            kx, ky = pts[min(t_idx, len(pts)-1)]
            # drejtim lokal
            if t_idx < len(pts) - 1:
                nx2, ny2 = pts[t_idx+1]
                angle = math.degrees(math.atan2(-(ny2-ky), nx2-kx))
            else:
                angle = 0
            surf = pygame.Surface((22, 10), pygame.SRCALPHA)
            pygame.draw.rect(surf, C["yellow"], (0, 0, 16, 8), border_radius=2)
            pygame.draw.rect(surf, (70, 70, 70), (16, 1, 6, 6), border_radius=1)
            rot = pygame.transform.rotate(surf, angle)
            s.blit(rot, rot.get_rect(center=(kx, ky)))

        # Shigjeta ne ~70% te kurbës
        if len(pts) > 2:
            idx70 = int(len(pts) * 0.68)
            idx72 = min(idx70 + 2, len(pts)-1)
            tip = pts[idx72]
            ax = tip[0] - pts[max(idx70-1,0)][0]
            ay = tip[1] - pts[max(idx70-1,0)][1]
            a = math.atan2(ay, ax)
            sz = 12
            ux2 = math.cos(a); uy2 = math.sin(a)
            atip  = (int(tip[0] + ux2*sz*0.5), int(tip[1] + uy2*sz*0.5))
            aleft = (int(tip[0] - sz*math.cos(a-0.55)), int(tip[1] - sz*math.sin(a-0.55)))
            aright= (int(tip[0] - sz*math.cos(a+0.55)), int(tip[1] - sz*math.sin(a+0.55)))
            pygame.draw.polygon(s, (0,0,0), [atip, aleft, aright])
            sz2 = sz-3
            atip2  = (int(tip[0] + ux2*sz2*0.5), int(tip[1] + uy2*sz2*0.5))
            aleft2 = (int(tip[0] - sz2*math.cos(a-0.5)), int(tip[1] - sz2*math.sin(a-0.5)))
            aright2= (int(tip[0] - sz2*math.cos(a+0.5)), int(tip[1] - sz2*math.sin(a+0.5)))
            pygame.draw.polygon(s, arrow_color, [atip2, aleft2, aright2])

    else:
        # Drejtvizore normale
        pygame.draw.line(s, (18, 24, 38), p1, p2, width + 6)
        pygame.draw.line(s, line_color, p1, p2, width)
        pygame.draw.line(s, (200, 220, 240), p1, p2, max(1, width // 6))

        if flowing and draw_truck:
            t = (tick * 0.008) % 1
            kx = int(x1 + dx * t)
            ky = int(y1 + dy * t)
            angle = math.degrees(math.atan2(-dy, dx))
            surf = pygame.Surface((22, 10), pygame.SRCALPHA)
            pygame.draw.rect(surf, C["yellow"], (0, 0, 16, 8), border_radius=2)
            pygame.draw.rect(surf, (70, 70, 70), (16, 1, 6, 6), border_radius=1)
            rot = pygame.transform.rotate(surf, angle)
            s.blit(rot, rot.get_rect(center=(kx, ky)))

        draw_arrow(s, p1, p2, arrow_color, width)



def edge_style(u, v, k_cur, i_cur, j_cur, phase):
    """
    Kthen (line_color, arrow_color, width, flowing, draw_truck) për çdo brinjë.

    Logjika:
    - update: i→j jeshil (ngjyra kryesore), i→k cyan, k→j portokalli
    - check / check_inf: i→k dhe k→j verdhë (s'ka update)
    - Kamioni lëviz vetëm mbi segmentet aktive i→k dhe k→j (nëse brinja ekziston)
    - check_inf: brinja gri e errët — s'ka rrugë
    """
    if phase in ("init", "done", "k_done"):
        return C["road"], C["arrow_default"], 3, False, False

    is_ik = (u == i_cur and v == k_cur)
    is_kj = (u == k_cur and v == j_cur)
    is_ij = (u == i_cur and v == j_cur)

    if phase == "update":
        if is_ij:
            return C["green"], C["arrow_update"], 7, False, False
        if is_ik:
            return C["cyan"], C["arrow_ik"], 6, True, True
        if is_kj:
            return C["orange"], C["arrow_kj"], 6, True, True

    if phase == "check":
        if is_ik:
            return C["yellow"], C["arrow_check"], 5, True, True
        if is_kj:
            return C["yellow"], C["arrow_check"], 5, True, True
        if is_ij:
            return (80, 100, 130), C["arrow_default"], 4, False, False

    if phase == "check_inf":
        if is_ik:
            return (50, 65, 90), C["arrow_inf"], 4, False, False
        if is_kj:
            return (50, 65, 90), C["arrow_inf"], 4, False, False

    return C["road"], C["arrow_default"], 3, False, False

# ─────────────────────── MATRICA ──────────────────────────────────
def draw_matrix(s, D, P, gw, H, k, i, j, phase, query_path):
    px = gw
    pw = s.get_width() - gw
    pad = 12

    txt(s, "Matrica  D[i][j]  (minuta)", 14, C["white"],
        (px + pw // 2, 16), center=True, bold=True)

    cell = min(38, (pw - pad * 2) // (N + 1))
    total_w = cell * (N + 1)
    ox = px + (pw - total_w) // 2
    oy = 36
    hlabels = ["D", "A", "B", "C", "1", "2", "3", "4"]

    for ci in range(N):
        cx = ox + (ci + 1) * cell
        r = pygame.Rect(cx + 1, oy + 1, cell - 2, cell - 2)
        pygame.draw.rect(s, (18, 32, 65), r, border_radius=3)
        pygame.draw.rect(s, C["cyan"], r, 1, border_radius=3)
        txt(s, hlabels[ci], 11, C["cyan"], (cx + cell//2, oy + cell//2), center=True, bold=True)

    for ri in range(N):
        ry2 = oy + (ri + 1) * cell
        r = pygame.Rect(ox + 1, ry2 + 1, cell - 2, cell - 2)
        pygame.draw.rect(s, (18, 32, 65), r, border_radius=3)
        pygame.draw.rect(s, C["cyan"], r, 1, border_radius=3)
        txt(s, hlabels[ri], 11, C["cyan"], (ox + cell//2, ry2 + cell//2), center=True, bold=True)

    for ri in range(N):
        for rj in range(N):
            cx2 = ox + (rj + 1) * cell
            cy2 = oy + (ri + 1) * cell
            r = pygame.Rect(cx2 + 1, cy2 + 1, cell - 2, cell - 2)

            if phase == "update" and ri == i and rj == j:
                bg, fc, brd = (15, 65, 25),  C["green"],    C["green"]
            elif phase in ("update","check") and ((ri==i and rj==k) or (ri==k and rj==j)):
                bg, fc, brd = (14, 45, 70),  C["cyan"],     C["cyan"]
            elif phase == "check_inf" and ((ri==i and rj==k) or (ri==k and rj==j)):
                bg, fc, brd = (20, 26, 40),  (75, 95, 125), (45, 60, 85)
            elif ri == k or rj == k:
                bg, fc, brd = (20, 33, 58),  (125,150,195), (35, 52, 82)
            elif ri == rj:
                bg, fc, brd = (10, 16, 32),  C["dim"],      (28, 40, 62)
            else:
                bg, fc, brd = (10, 15, 30),  C["dim"],      (28, 38, 60)

            if query_path and len(query_path) > 1:
                for qi in range(len(query_path) - 1):
                    if ri == query_path[qi] and rj == query_path[qi + 1]:
                        bg, fc, brd = (48, 15, 70), C["purple"], C["purple"]

            pygame.draw.rect(s, bg,  r, border_radius=3)
            pygame.draw.rect(s, brd, r, 1, border_radius=3)
            val = D[ri][rj]
            vstr = "inf" if val == INF else str(int(val))
            txt(s, vstr, 12, fc, (cx2 + cell//2, cy2 + cell//2),
                center=True, bold=(val < INF))

    matrix_bottom = oy + (N + 1) * cell

    ly = matrix_bottom + 8
    pygame.draw.line(s, (35, 50, 80), (px + pad, ly), (px + pw - pad, ly))
    ly += 7
    txt(s, "LEGJENDA", 11, C["dim"], (px + pw//2, ly + 1), center=True, bold=True)
    ly += 15

    legend_items = [
        (C["green"],   (15, 65, 25),  "UPDATE — rruge e re me e shkurter"),
        (C["cyan"],    (14, 45, 70),  "i->k dhe k->j (segmentet aktive)"),
        (C["yellow"],  (45, 40, 10),  "CHECK — pa permiresim"),
        ((75,95,125),  (20, 26, 40),  "INF — s'ka rruge nepermjet k"),
        (C["purple"],  (48, 15, 70),  "QUERY — rruga e kerkuar"),
    ]
    for brd_c, bg_c, label in legend_items:
        item_r = pygame.Rect(px + pad, ly, pw - pad*2, 19)
        pygame.draw.rect(s, bg_c,  item_r, border_radius=4)
        pygame.draw.rect(s, brd_c, item_r, 1, border_radius=4)
        pygame.draw.rect(s, brd_c, (px + pad + 4, ly + 4, 11, 11), border_radius=2)
        txt(s, label, 10, (210, 225, 245), (px + pad + 20, ly + 4))
        ly += 22

    return ly

# ─────────────────────── APIKACIONI KRYESOR ───────────────────────
class App:
    def __init__(self):
        self.windowed = (1440, 860)
        self.full = False
        self.screen = pygame.display.set_mode(self.windowed, pygame.RESIZABLE)
        pygame.display.set_caption("Floyd-Warshall — UrbanPost Sistemi Logjistik")
        self.clock = pygame.time.Clock()
        self.tick = 0
        self.paused = True
        self.counter = 0
        self.speed = 2
        self.query_src = 0
        self.query_dst = 7
        self.show_query = False
        self.reset()

    def reset(self):
        self.steps, self.final_D, self.final_P = floyd_warshall_steps()
        self.step_idx = 0
        self.logs = []
        self.cur = self.steps[0]
        self.log("══ UrbanPost — Floyd-Warshall Fillon ══", C["blue"])
        self.log(f"Rrjet: {N} nyje, {len(EDGES_RAW)} rrugë të drejtuara.", C["dim"])
        self.log("D[i][j] = koha direkte (min) ose INF nese s'ka lidhje.", C["dim"])
        self.log("Algoritmi gjen rruget me te shkurtra ndermjet", C["dim"])
        self.log("CDO cifti nyjesh — Depo, QR dhe PD.", C["dim"])
        self.log("SPACE: luaj/pauzo | S: hap | G: shfaq rrugën", C["dim"])

    def log(self, msg, col):
        self.logs.append((msg, col))
        self.logs = self.logs[-20:]

    def node_pos(self):
        W, H = self.graph_w, self.screen.get_height()
        top = 95; bot = H - 65; span = bot - top
        return {
            0: (int(W * 0.10),  int(top + span * 0.50)),
            1: (int(W * 0.36),  int(top + span * 0.14)),
            2: (int(W * 0.36),  int(top + span * 0.50)),
            3: (int(W * 0.36),  int(top + span * 0.86)),
            4: (int(W * 0.73),  int(top + span * 0.10)),
            5: (int(W * 0.73),  int(top + span * 0.37)),
            6: (int(W * 0.73),  int(top + span * 0.63)),
            7: (int(W * 0.73),  int(top + span * 0.90)),
        }

    def apply_step(self):
        if self.step_idx >= len(self.steps) - 1:
            return
        self.step_idx += 1
        self.cur = self.steps[self.step_idx]
        kind, D, P, k, i, j, extra = self.cur

        # Emrat e nyjeve për log më të lexueshëm
        def nm(idx):
            labels = ["Depo","QR-A","QR-B","QR-C","PD-1","PD-2","PD-3","PD-4"]
            return labels[idx]

        if kind == "update":
            old, new = extra
            old_s = "INF" if old == INF else f"{int(old)}min"
            # Shpjego kuptimin praktik
            if i == 0:
                src_desc = "Depoja"
            elif i in (1,2,3):
                src_desc = f"QR nga {nm(i)}"
            else:
                src_desc = f"PD nga {nm(i)}"
            self.log(
                f"UPDATE D[{SHORT[i]}][{SHORT[j]}]: {old_s}->{int(new)}min",
                C["green"])
            self.log(
                f"  Via {nm(k)}: {nm(i)}->{nm(k)}->{nm(j)} eshte me e shkurter!",
                C["cyan"])

        elif kind == "check":
            cur_val = D[i][j]
            via_val = D[i][k] + D[k][j]
            cur_s = "INF" if cur_val == INF else f"{int(cur_val)}min"
            self.log(
                f"CHECK [{SHORT[i]}]->[{SHORT[j]}] via {nm(k)}: "
                f"{int(via_val)}min >= {cur_s} — pa ndryshim",
                C["yellow"])

        elif kind == "check_inf":
            # Trego cilin segment mungon
            ik_inf = D[i][k] == INF
            kj_inf = D[k][j] == INF
            if ik_inf and kj_inf:
                reason = f"{nm(i)}->{nm(k)} dhe {nm(k)}->{nm(j)} = INF"
            elif ik_inf:
                reason = f"{nm(i)}->{nm(k)} = INF"
            else:
                reason = f"{nm(k)}->{nm(j)} = INF"
            self.log(
                f"INF [{SHORT[i]}]->[{SHORT[j]}] via {nm(k)}: {reason}",
                (70, 85, 110))

        elif kind == "k_done":
            if extra:
                self.log(f"GABIM: Cikel negativ pas k={nm(k)}!", C["red"])
            else:
                # Sa update u bënë gjatë këtij k
                updates_for_k = sum(
                    1 for st in self.steps[:self.step_idx+1]
                    if st[0] == "update" and st[3] == k
                )
                self.log(
                    f"-- k={SHORT[k]} ({nm(k)}) kompletuar --",
                    C["active_k"])
                if updates_for_k > 0:
                    self.log(
                        f"   {updates_for_k} rruge u permiresuan duke kaluar {nm(k)}.",
                        C["green"])
                else:
                    self.log(
                        f"   Asnje permiresum — {nm(k)} nuk ishte i dobishem.",
                        C["dim"])

        elif kind == "done":
            # Llogarit statistikat finale
            total_updates = sum(1 for st in self.steps if st[0] == "update")
            reachable = sum(
                1 for r in range(N) for c in range(N)
                if r != c and self.final_D[r][c] < INF
            )
            best = min(
                (self.final_D[r][c], r, c)
                for r in range(N) for c in range(N)
                if r != c and self.final_D[r][c] < INF
            )
            worst = max(
                (self.final_D[r][c], r, c)
                for r in range(N) for c in range(N)
                if r != c and self.final_D[r][c] < INF
            )
            self.log("══ FLOYD-WARSHALL PERFUNDOI ══", C["green"])
            self.log(f"Perditesime totale: {total_updates} rruge te optimizuara.", C["green"])
            self.log(f"Cefte te arritshme: {reachable} nga {N*(N-1)} te mundshme.", C["cyan"])
            self.log(
                f"Rruga me e shkurter: {nm(best[1])}->{nm(best[2])} = {int(best[0])}min",
                C["cyan"])
            self.log(
                f"Rruga me e gjate:   {nm(worst[1])}->{nm(worst[2])} = {int(worst[0])}min",
                C["dim"])
            self.log("Algoritmi vleu: tani UrbanPost di rrugen", C["blue"])
            self.log("optimale nga CDO pike deri CDO pike ne rrjet!", C["blue"])
            self.log("Shtyp G per te shfaqur rrugën e deshiruar.", C["dim"])
            self.paused = True

    def draw_graph(self):
        s = self.screen
        W, H = self.graph_w, s.get_height()
        pygame.draw.rect(s, C["graph"], (12, 80, W - 24, H - 130), border_radius=16)

        pos = self.node_pos()
        kind, D, P, k, i, j, extra = self.cur

        # ── Vizato harqet ──
        for u, v, w in EDGES_RAW:
            lc, ac, width, flowing, draw_truck = edge_style(u, v, k, i, j, kind)
            # Vetëm vizato kamionin nëse brinja fizikisht ekziston
            draw_road(s, pos[u], pos[v], lc, ac, width, self.tick,
                      flowing=flowing, draw_truck=draw_truck, u=u, v=v)

        # Rruga e query mbi harqet
        qp = None
        if self.show_query:
            qp = reconstruct(self.final_P, self.query_src, self.query_dst)
            if qp and len(qp) > 1:
                for qi in range(len(qp) - 1):
                    pygame.draw.line(s, C["purple"],
                                     pos[qp[qi]], pos[qp[qi + 1]], 6)

        # ── Vizato nyjet ──
        for ni in range(N):
            x, y = pos[ni]
            is_k  = (ni == k and kind not in ("init", "done"))
            is_ij = (ni in (i, j) and kind in ("update", "check", "check_inf"))
            active = is_k or is_ij or (qp and ni in qp)

            if ni == 0:
                draw_depo(s, x, y, active, self.tick)
                txt(s, "DEPO QENDRORE", 15, C["depo"], (x, y + 80), center=True, bold=True)
                txt(s, "Pikënisjeje", 12, C["dim"], (x, y + 98), center=True)
            elif ni in (1, 2, 3):
                draw_qr(s, x, y, SHORT[ni], active, self.tick)
                txt(s, "Qendër Rajonale", 12, C["dim"], (x, y + 68), center=True)
            else:
                draw_pd(s, x, y, SHORT[ni], active)

        # Etiketa ndërmjetës k
        if k >= 0 and kind not in ("done",):
            kx, ky = pos[k]
            r = pygame.Rect(kx - 40, ky - 90, 80, 26)
            pygame.draw.rect(s, (50, 40, 10), r, border_radius=6)
            pygame.draw.rect(s, C["active_k"], r, 2, border_radius=6)
            txt(s, f"k = {SHORT[k]}", 14, C["active_k"], r.center, center=True, bold=True)

        # Peshat e harqeve — me pozicion mbi kurbë
        for u, v, w in EDGES_RAW:
            x1, y1 = pos[u]; x2, y2 = pos[v]
            ctrl, curved = get_ctrl_point(pos[u], pos[v], u, v)
            if curved:
                # Midpoint e kurbës Bezier (t=0.5)
                cx3 = int(0.25*x1 + 0.5*ctrl[0] + 0.25*x2)
                cy3 = int(0.25*y1 + 0.5*ctrl[1] + 0.25*y2)
                # Zhvendos pak nga kurba
                dx2, dy2 = cx3 - (x1+x2)//2, cy3 - (y1+y2)//2
                L2 = math.hypot(dx2, dy2) or 1
                lx = cx3 + int(dx2/L2 * 10)
                ly = cy3 + int(dy2/L2 * 10)
            else:
                mx, my = (x1 + x2) // 2, (y1 + y2) // 2
                dx2, dy2 = x2 - x1, y2 - y1
                L2 = math.hypot(dx2, dy2) or 1
                nx_, ny_ = -dy2 / L2, dx2 / L2
                lx, ly = int(mx + nx_ * 22), int(my + ny_ * 22)
            r2 = pygame.Rect(lx - 22, ly - 11, 44, 21)
            pygame.draw.rect(s, (4, 8, 18), r2, border_radius=5)
            pygame.draw.rect(s, C["road_hi"], r2, 1, border_radius=5)
            txt(s, f"{w}min", 12, C["white"], r2.center, center=True)

        # Titull compact
        pygame.draw.rect(s, (6, 10, 24), (12, 4, W - 24, 76), border_radius=10)
        txt(s, "FLOYD-WARSHALL — UrbanPost Sistemi Logjistik",
            22, C["white"], (W // 2, 24), center=True, bold=True)
        txt(s, "Depo Qendrore  →  Qendra Rajonale (QR)  →  Pika Dorëzimi (PD)",
            13, C["blue"], (W // 2, 48), center=True)

        # Bari i statusit — me ngjyre sipas fazave
        status = self._status_msg()
        bar_col = {
            "update": (12, 55, 18), "check": (50, 42, 8),
            "check_inf": (12, 16, 36), "k_done": (48, 40, 8),
            "done": (10, 52, 18), "init": (10, 20, 50),
        }.get(kind, (8, 12, 28))
        pygame.draw.rect(s, bar_col, (12, H - 46, W - 24, 36), border_radius=8)
        pygame.draw.rect(s, self._status_color(),
                         (12, H - 46, W - 24, 36), 1, border_radius=8)
        txt(s, status, 13, C["white"], (W // 2, H - 28), center=True)

    def _status_msg(self):
        kind, D, P, k, i, j, extra = self.cur
        if kind == "init":
            return "Inicializim. Shtyp SPACE ose S për të filluar."
        if kind == "done":
            return "✔ Floyd-Warshall përfundoi. Të gjitha rrugët minimale janë llogaritur!"
        if kind == "k_done":
            return f"── Ndërmjetësi k = {SHORT[k]} u kompletua ──"
        if kind == "update":
            old, new = extra
            old_s = "∞" if old == INF else str(int(old))
            return f"[UPDATE] D[{SHORT[i]}][{SHORT[j]}]: {old_s} → {int(new)} nëpër k={SHORT[k]}"
        if kind == "check":
            return f"[CHECK] D[{SHORT[i]}][{SHORT[j]}] via k={SHORT[k]}: rrugë ekziston por pa përmirësim"
        if kind == "check_inf":
            return f"[INF] D[{SHORT[i]}][{SHORT[j]}] via k={SHORT[k]}: D[i][k] ose D[k][j] = ∞ — kalohet"
        return ""

    def draw_panel(self):
        s = self.screen
        W_full = s.get_width()
        H = s.get_height()
        px = self.graph_w
        pw = W_full - px

        # Background panel
        pygame.draw.rect(s, (8, 13, 28),  (px, 0, pw, H))
        pygame.draw.rect(s, (14, 22, 46), (px + 6, 6, pw - 12, H - 12), border_radius=12)

        kind, D, P, k, i, j, extra = self.cur

        # ── Zona 1: Matrica + Legjenda (lart) ──
        leg_bottom = draw_matrix(s, D, P, px, H, k, i, j, kind,
                    reconstruct(self.final_P, self.query_src, self.query_dst)
                    if self.show_query else None)

        # ── Zona 2: Info aktuale (bllok i dukshëm nën legjendë) ──
        info_y = leg_bottom + 8
        pad = 10

        # Blloku i statusit aktual — me background
        status_txt = self._status_short()
        info_r = pygame.Rect(px + pad, info_y, pw - pad*2, 44)
        pygame.draw.rect(s, (12, 20, 48), info_r, border_radius=6)
        pygame.draw.rect(s, self._status_color(), info_r, 2, border_radius=6)
        # Lloji i hapit — badge
        kind_label = {"init":"INIT","update":"UPDATE","check":"CHECK",
                      "check_inf":"INF","k_done":"K DONE","done":"DONE"}.get(kind, kind.upper())
        badge_c = self._status_color()
        badge_r = pygame.Rect(px + pad + 6, info_y + 6, 58, 16)
        pygame.draw.rect(s, badge_c, badge_r, border_radius=3)
        txt(s, kind_label, 10, (5, 8, 18), badge_r.center, center=True, bold=True)
        # Teksti i hapit
        txt(s, status_txt, 12, (220, 235, 255), (px + pad + 8, info_y + 26))

        # Hapi aktual / total — djathtas
        total = len(self.steps)
        step_str = f"{self.step_idx + 1}/{total}"
        txt(s, step_str, 11, C["dim"], (px + pw - pad - 8, info_y + 6))

        info_bottom = info_y + 52

        # ── Zona 3: Log i ekzekutimit ──
        log_y = info_bottom + 6
        pygame.draw.line(s, (30, 45, 75), (px + pad, log_y), (px + pw - pad, log_y))
        log_y += 5
        txt(s, "LOG EKZEKUTIMI", 11, C["dim"], (px + pad + 4, log_y), bold=True)
        log_y += 15

        max_chars = max(24, (pw - pad*2 - 8) // 7)
        for msg, col in self.logs[-10:]:
            # Background per cdo log entry
            log_r = pygame.Rect(px + pad, log_y, pw - pad*2, 15)
            pygame.draw.rect(s, (10, 16, 34), log_r)
            # Clip text
            display = msg if len(msg) <= max_chars else msg[:max_chars-2] + ".."
            txt(s, display, 11, col, (px + pad + 4, log_y + 1))
            log_y += 16
            if log_y > H - 110:
                break

        # ── Zona 4: Query + Controls (poshtë) ──
        ctrl_y = H - 100
        pygame.draw.line(s, (30, 45, 75), (px + pad, ctrl_y), (px + pw - pad, ctrl_y))
        ctrl_y += 6

        # Query
        src_lbl = ["Depo","QR-A","QR-B","QR-C","PD-1","PD-2","PD-3","PD-4"][self.query_src]
        dst_lbl = ["Depo","QR-A","QR-B","QR-C","PD-1","PD-2","PD-3","PD-4"][self.query_dst]
        q_r = pygame.Rect(px + pad, ctrl_y, pw - pad*2, 32)
        pygame.draw.rect(s, (14, 18, 40), q_r, border_radius=5)
        pygame.draw.rect(s, C["purple"], q_r, 1, border_radius=5)
        txt(s, f"QUERY  {src_lbl} -> {dst_lbl}", 11, C["purple"], (px + pad + 8, ctrl_y + 4), bold=True)

        if self.show_query:
            qp = reconstruct(self.final_P, self.query_src, self.query_dst)
            if kind == "done" and qp:
                d = self.final_D[self.query_src][self.query_dst]
                path_str = " -> ".join(["D","A","B","C","1","2","3","4"][n] for n in qp)
                txt(s, f"{path_str}  =  {int(d)} min", 11, C["green"],
                    (px + pad + 8, ctrl_y + 18), bold=True)
            elif kind != "done":
                txt(s, "shtyp G pas perfundimit per te pare rrugen", 10, C["dim"],
                    (px + pad + 8, ctrl_y + 18))
        else:
            txt(s, "G: aktivizo / fsheh rrugen", 10, C["dim"], (px + pad + 8, ctrl_y + 18))

        ctrl_y += 38
        # Tastet
        controls = [
            "SPACE: luaj/pauzo    S/->: hap    R: rinicio",
            "Q/W: burimi    E/D: destinacioni    G: query",
        ]
        for ln in controls:
            txt(s, ln, 10, (80, 100, 140), (px + pad + 4, ctrl_y))
            ctrl_y += 14

    def _status_color(self):
        kind = self.cur[0]
        return {
            "update": C["green"], "check": C["yellow"],
            "check_inf": (60, 75, 110), "k_done": C["active_k"],
            "done": C["green"], "init": C["blue"],
        }.get(kind, C["dim"])

    def _status_short(self):
        kind, D, P, k, i, j, extra = self.cur
        nm = ["Depo","QR-A","QR-B","QR-C","PD-1","PD-2","PD-3","PD-4"]
        if kind == "init":   return "Inicializim — matrica me peshat fillestare"
        if kind == "done":   return "Perfundoi! Te gjitha rruget u llogariten."
        if kind == "k_done": return f"k={SHORT[k]} ({nm[k]}) kompletuar"
        if kind == "update":
            old, new = extra
            os2 = "INF" if old == INF else f"{int(old)}min"
            return f"D[{SHORT[i]}][{SHORT[j]}]: {os2} -> {int(new)}min  via {nm[k]}"
        if kind == "check":
            via = D[i][k] + D[k][j]
            cur = D[i][j]
            cs2 = "INF" if cur == INF else f"{int(cur)}min"
            return f"D[{SHORT[i]}][{SHORT[j]}]: {int(via)}min >= {cs2}  no update"
        if kind == "check_inf":
            return f"D[{SHORT[i]}][{SHORT[j]}] via {nm[k]}: segmenti = INF"
        return ""

    def draw(self):
        self.screen.fill(C["bg"])
        self.draw_graph()
        self.draw_panel()
        pygame.display.flip()

    def resize(self, w, h):
        self.graph_w = int(w * 0.62)

    def toggle_full(self):
        self.full = not self.full
        if self.full:
            self.screen = pygame.display.set_mode((0, 0), pygame.FULLSCREEN)
        else:
            self.screen = pygame.display.set_mode(self.windowed, pygame.RESIZABLE)
        self.resize(*self.screen.get_size())

    def events(self):
        for e in pygame.event.get():
            if e.type == pygame.QUIT:
                return False
            if e.type == pygame.VIDEORESIZE and not self.full:
                self.windowed = (e.w, e.h)
                self.screen = pygame.display.set_mode(self.windowed, pygame.RESIZABLE)
                self.resize(e.w, e.h)
            if e.type == pygame.KEYDOWN:
                if e.key == pygame.K_ESCAPE:
                    return False
                elif e.key == pygame.K_SPACE:
                    self.paused = not self.paused
                elif e.key in (pygame.K_s, pygame.K_RIGHT):
                    self.apply_step()
                elif e.key == pygame.K_r:
                    self.reset()
                elif e.key in (pygame.K_f, pygame.K_F11):
                    self.toggle_full()
                elif e.key == pygame.K_q:
                    self.query_src = (self.query_src - 1) % N
                elif e.key == pygame.K_w:
                    self.query_src = (self.query_src + 1) % N
                elif e.key == pygame.K_e:
                    self.query_dst = (self.query_dst - 1) % N
                elif e.key == pygame.K_d:
                    self.query_dst = (self.query_dst + 1) % N
                elif e.key == pygame.K_g:
                    self.show_query = not self.show_query
        return True

    def run(self):
        self.resize(*self.windowed)
        running = True
        while running:
            self.tick += 1
            running = self.events()
            if not self.paused:
                self.counter += 1
                if self.counter >= max(1, 60 // self.speed):
                    self.apply_step()
                    self.counter = 0
            self.draw()
            self.clock.tick(60)
        pygame.quit()
        sys.exit()

# ─────────────────────── STATISTIKAT FINALE ───────────────────────
def print_results(D, P):
    labels = ["Depo","QR-A","QR-B","QR-C","PD-1","PD-2","PD-3","PD-4"]

    print("\n" + "=" * 70)
    print("  FLOYD-WARSHALL — REZULTATET FINALE  (UrbanPost Sistemi Logjistik)")
    print("=" * 70)

    for v in range(N):
        if D[v][v] < 0:
            print(f"\n  GABIM: Cikël negativ i zbuluar në kulmin {SHORT[v]}!")
            return

    # ── Matrica ──
    print("\n  Matrica e Distancave Minimale D[i][j] (minuta):\n")
    header = "           " + "  ".join(f"{s:>6}" for s in SHORT)
    print(header)
    print("           " + "-" * (len(header) - 11))
    for ri in range(N):
        row = f"  {labels[ri]:<8} |"
        for rj in range(N):
            val = D[ri][rj]
            row += f"  {'INF':>6}" if val == INF else f"  {int(val):>6}"
        print(row)

    # ── Kompleksiteti ──
    print(f"\n  Kompleksiteti: O(V^3) = O({N}^3) = O({N**3}) operacione")
    print(f"  Iterime totale (duke perfshire INF-kontrollet): {N**3} + {N} k_done = {N**3+N} hapa")

    # ── Rrugët optimale ──
    print("\n  Rruget Optimale ndermjet te gjitha cifteve:\n")
    for src in range(N):
        for dst in range(N):
            if src != dst and D[src][dst] < INF:
                path = reconstruct(P, src, dst)
                path_str = " -> ".join(labels[n] for n in path)
                print(f"    {labels[src]:<8} -> {labels[dst]:<8}: {int(D[src][dst]):>3} min  |  {path_str}")

    # ── Statistika interesante ──
    print("\n" + "=" * 70)
    print("  PERFUNDIM: CFARE NA VLEU ALGORITMI?")
    print("=" * 70)

    reachable = [(r, c) for r in range(N) for c in range(N)
                 if r != c and D[r][c] < INF]
    unreachable = [(r, c) for r in range(N) for c in range(N)
                   if r != c and D[r][c] == INF]

    all_dists = [D[r][c] for r, c in reachable]
    best_d, best_pair = min((D[r][c], (r,c)) for r,c in reachable)
    worst_d, worst_pair = max((D[r][c], (r,c)) for r,c in reachable)
    avg_d = sum(all_dists) / len(all_dists)

    print(f"\n  Cefte te arritshme: {len(reachable)} nga {N*(N-1)} te mundshme.")
    if unreachable:
        print(f"  Cefte te PA-arritshme: {len(unreachable)} (shkak: grafiku jo i plote).")

    print(f"\n  Rruga me e shkurter: {labels[best_pair[0]]} -> {labels[best_pair[1]]}"
          f" = {int(best_d)} min")
    print(f"  Rruga me e gjate:   {labels[worst_pair[0]]} -> {labels[worst_pair[1]]}"
          f" = {int(worst_d)} min")
    print(f"  Koha mesatare e dergeses: {avg_d:.1f} min")

    print("\n  Vlerat e perditësuara tregojne:")
    print("    - Rruge indirekte (nepermjet nyje te ndermjetme) jane me te shkurtera")
    print("      se lidhja direkte ne shume raste.")
    print("    - P.sh. per te shkuar nga Depoja ne PD-4: rruga direkte mungon,")
    d_0_7 = D[0][7]
    p_0_7 = reconstruct(P, 0, 7)
    path_str_07 = " -> ".join(labels[n] for n in p_0_7) if p_0_7 else "N/A"
    print(f"      por {path_str_07} = {int(d_0_7)} min.")

    # Sa update u bënë gjatë algoritmit
    # (duhet ri-ekzekutuar logjiken per kete)
    D2, P2 = build_matrix()
    updates = []
    for k in range(N):
        for ii in range(N):
            for jj in range(N):
                if D2[ii][k] < INF and D2[k][jj] < INF:
                    nv = D2[ii][k] + D2[k][jj]
                    if nv < D2[ii][jj]:
                        old_v = D2[ii][jj]
                        updates.append((k, ii, jj, old_v, nv))
                        D2[ii][jj] = nv
                        P2[ii][jj] = P2[k][jj]

    print(f"\n  Algoritmi beri {len(updates)} perditesime te matricës gjatë ekzekutimit.")
    print("  Secili perditesim = zbulim i nje rruge me te shkurter nepermjet nje")
    print("  nyje ndermjetese — si te gjesh 'shkorrcen' me te mire per kamionin.")

    print("\n  Konkluzion praktik per UrbanPost:")
    print("    Pas ekzekutimit, menaxheri di:")
    print("    a) Sa minuta duhen nga CDO pike deri CDO pike tjeter.")
    print("    b) Cilin itinerar te ndjeke CDO kamion.")
    print("    c) Nese nje rruge bllokohet, mund te rillogarite.")
    print("=" * 70 + "\n")


if __name__ == "__main__":
    _, D, P = floyd_warshall_steps()
    print_results(D, P)
    App().run()


  FLOYD-WARSHALL — REZULTATET FINALE  (UrbanPost Sistemi Logjistik)

  Matrica e Distancave Minimale D[i][j] (minuta):

                0       1       2       3       4       5       6       7
           --------------------------------------------------------------
  Depo     |       0      12      18      22      27      29      35      41
  QR-A     |     INF       0       7      16      15      18      24      32
  QR-B     |     INF       7       0       9      17      11      17      25
  QR-C     |     INF      16       9       0      26      20      13      19
  PD-1     |     INF     INF     INF     INF       0       6      14      20
  PD-2     |     INF     INF     INF     INF       6       0       8      14
  PD-3     |     INF     INF     INF     INF      14       8       0      10
  PD-4     |     INF     INF     INF     INF      24      18      10       0

  Kompleksiteti: O(V^3) = O(8^3) = O(512) operacione
  Iterime totale (duke perfshire INF-kontrollet): 512 + 8 k_d

SystemExit: 

---

## Ushtrimi 2 — Rrjeti i Klinikave dhe Transportit të Urgjencës Mjekësore

-----------------------------------------------------------------------------

### Skenari Real

Sistemi shëndetësor i një rajoni ka **spitale**, **klinika** dhe **stacione ambulance**
të shpërndarë nëpër qytet. Kur ndodh një urgjencë, sistemi duhet të dijë menjëherë:

- *"Cili është stacioni i ambulancës më afër rastit?"*
- *"Cila klinikë mund të arrihet në kohën minimale nga spitali qendror?"*
- *"Si të ridrejtojmë urgjencën nëse rruga kryesore është e zënë?"*

Çdo lidhje (rrugë) ka një **kohë mesatare lëvizjeje** e peshuar me trafik, ndriçim
dhe lloj rruge. Sistemi duhet t'i ketë **të gjitha distancat minimale të llogaritura paraprakisht** —
kur vjen thirrja e urgjencës, t'i japë menjëherë rrugën optimale pa vonesë.

-----------------------------------------------------------------------------

### Pse Floyd-Warshall për Rrjete Mjekësore?

#### Në situata urgjence:

- Ambulanca mund të niset nga **çdo stacion** — jo vetëm nga spitali qendror
- Raste urgjente ndodhin **kudo** dhe duhet rruga optimale **pa vonesat e rikompjutimit**
- Disa rrugë janë **njëkaheshe** (rrugë të sheshta, tunele me sens të vetëm)
- Matrica e plotë D[i][j] mundëson **ridrejtim të menjëhershëm** nëse bllokohet një rrugë
-----------------------------------------------------------------------------

### Struktura e Rrjetit

```
SPITALI QENDROR ──→ [SA-1, SA-2, SA-3] ──→ [KR-A, KR-B] ──→ [QU-1, QU-2]
      (SQ)           (Stacione Amb.)       (Klinika Rajonale)  (Q.Urgjence)
```

**Detajet e grafit:**
- **8 kulme:** SQ (1) + Stacione Ambulancës (3) + Klinika Rajonale (2) + Qendra Urgjence (2)
- **19 harqe** — disa **njëkaheshe** (p.sh. QU-1→SA-1 ekziston, jo anasjelltas)
- **Graf i drejtuar jo-simetrik:** pasqyron rrugët reale me sens të vetëm
- **Peshat:** 5–49 minuta kohë lëvizjeje

### Çfarë Simulojmë?

Animimi tregon hap pas hapi si Floyd-Warshall ndërton **matricën e plotë të distancave**
duke provuar sistematikisht çdo kulm si ndërmjetës. Paneli djathtas shfaq:

| Element i Panelit | Kuptimi |
|-------------------|---------|
| **Matrica D[i][j]** | Distanca aktuale e njohur midis çdo çifti |
| **Bar-chart aksesueshmërie** | Σ distancave nga çdo nyje — identifikon qendrën optimale |
| **Regjistri i hapave** | Çdo përditësim D[i][j] me vlerat para/pas |
| **Ngjyrat e nyjeve** | 🔴 Spital, 🟡 Ambulancë, 🟢 Klinikë, 🟣 Urgjencë |

### Si ta Arrijmë?

1. **Graf i drejtuar jo-simetrik:** D[i][j] ≠ D[j][i] — pasqyron rrugë njëkaheshe
2. **Snapshots:** çdo herë që D[i][j] përmirësohet, ruhet një hap animimi
3. **Identifikimi i qendrës:** pas llogaritjes, gjendet `argmin_v Σ_j D[v][j]`
4. **Detektimi i cikleve negative:** pas çdo k, kontrollohet `D[v][v] < 0`

**Tastet gjatë simulimit:**
- `SPACE` — luaj/pauzo | `S` — hap i vetëm | `R` — rinicio
- `F11` — ekran i plotë | `ESC` — dil


### Implementimi i Ushtrimit 2 — Rrjeti Mjekësor


In [9]:
"""
Ushtrimi 2 - Floyd-Warshall: Rrjeti i Klinikave dhe Transportit te Urgjences Mjekesore
Spital Qendror, Stacione Ambulance, Klinika Rajonale, Qendra Urgjence

NDRYSHIMET (te njejtat si Ushtrimi 1):
- Ambulancat levizin VETEM mbi brinjetet aktive (i->k dhe k->j)
- Brinjete kurbezuara me Bezier per brinjetet qe kapercejne nyje ne te njejten kolone
- Algoritmi iterOn GJITHMONW per cdo (i,j) - edhe kur INF (check_inf faza)
- Layout i panelit i ridesajnuar: Matrica + Legjenda + Log + Query + Controls
- Loget specifike me emra nyjesh dhe kontekst mjekesor
- Permbledhje finale e qarte
"""

import pygame, sys, math
from copy import deepcopy

pygame.init()

# ───────────────────────── PALETA MJEKESORE ──────────────────────────
C = {
    "bg":           (5, 8, 20),
    "graph":        (8, 14, 30),
    "panel":        (10, 18, 38),
    "panel2":       (15, 25, 52),
    "white":        (235, 248, 255),
    "dim":          (105, 132, 175),
    "blue":         (60, 180, 255),
    "cyan":         (30, 210, 245),
    "yellow":       (245, 195, 40),
    "red":          (255, 70, 55),
    "green":        (50, 210, 110),
    "orange":       (255, 152, 50),
    "purple":       (165, 115, 255),
    "road":         (50, 68, 92),
    "road_hi":      (90, 120, 160),
    "hospital":     (230, 50,  65),
    "ambulance":    (255, 200, 40),
    "clinic":       (40,  175, 130),
    "urgency":      (200, 80,  220),
    "inactive":     (42,  55,  78),
    "active_k":     (255, 210, 55),
    # Ngjyrat e shigjetave
    "arrow_default":(130, 160, 205),
    "arrow_update": (50,  210, 110),
    "arrow_check":  (245, 195, 40),
    "arrow_inf":    (70,  88,  115),
    "arrow_ik":     (30,  210, 245),
    "arrow_kj":     (255, 152, 50),
}

INF = float('inf')
FONTS = {}

# ───────────────────────── GRAFIKU MJEKESOR ──────────────────────────
NODE_NAMES = [
    "Spitali Qendror",   # 0  SQ
    "SA-1",              # 1  Stacioni Ambulances 1
    "SA-2",              # 2  Stacioni Ambulances 2
    "SA-3",              # 3  Stacioni Ambulances 3
    "KR-A",              # 4  Klinika Rajonale A
    "KR-B",              # 5  Klinika Rajonale B
    "QU-1",              # 6  Qendra Urgjence 1
    "QU-2",              # 7  Qendra Urgjence 2
]
SHORT  = ["SQ",  "SA1", "SA2", "SA3", "KRA", "KRB", "QU1", "QU2"]
LABELS = ["SQ","SA-1","SA-2","SA-3","KR-A","KR-B","QU-1","QU-2"]
NTYPE  = ["hospital","ambulance","ambulance","ambulance",
          "clinic","clinic","urgency","urgency"]
N = len(SHORT)

EDGES_RAW = [
    (0, 1,  8), (0, 2, 12), (0, 3, 15),
    (1, 0,  8), (1, 4, 10), (1, 6, 18),
    (2, 0, 12), (2, 1,  6), (2, 4, 14), (2, 5, 11),
    (3, 5,  9), (3, 7, 13),
    (4, 6,  7), (4, 7, 16),
    (5, 4,  5), (5, 7,  8),
    (6, 7,  6), (6, 1, 20),
    (7, 3, 17),
]

# ───────────────────────── FONT HELPER ───────────────────────────────
def fnt(size, bold=False):
    k = (size, bold)
    if k not in FONTS:
        FONTS[k] = pygame.font.SysFont("Segoe UI", size, bold=bold)
    return FONTS[k]

def txt(s, msg, size, col, pos, center=False, bold=False):
    img = fnt(size, bold).render(str(msg), True, col)
    r = img.get_rect(center=pos) if center else img.get_rect(topleft=pos)
    s.blit(img, r)
    return r

def glow(s, x, y, col, r):
    for rr in range(r, 0, -14):
        a = int(70 * rr / r)
        g = pygame.Surface((rr*2, rr*2), pygame.SRCALPHA)
        pygame.draw.circle(g, (*col, a), (rr, rr), rr)
        s.blit(g, (x-rr, y-rr))

# ───────────────────────── FLOYD-WARSHALL ────────────────────────────
def build_matrix():
    D = [[INF]*N for _ in range(N)]
    P = [[None]*N for _ in range(N)]
    for i in range(N): D[i][i] = 0
    for u, v, w in EDGES_RAW:
        D[u][v] = w
        P[u][v] = u
    return D, P

def floyd_warshall_steps():
    """
    Iteron GJITHMONW per cdo (i,j,k) — edhe kur INF.
    Tre faza: update / check / check_inf
    """
    D, P = build_matrix()
    steps = [("init", deepcopy(D), deepcopy(P), -1, -1, -1, None)]

    for k in range(N):
        for i in range(N):
            for j in range(N):
                if D[i][k] < INF and D[k][j] < INF:
                    nv = D[i][k] + D[k][j]
                    if nv < D[i][j]:
                        old = D[i][j]
                        D[i][j] = nv
                        P[i][j] = P[k][j]
                        steps.append(("update", deepcopy(D), deepcopy(P),
                                      k, i, j, (old, nv)))
                    else:
                        steps.append(("check", deepcopy(D), deepcopy(P),
                                      k, i, j, None))
                else:
                    steps.append(("check_inf", deepcopy(D), deepcopy(P),
                                  k, i, j, None))
        neg = any(D[v][v] < 0 for v in range(N))
        steps.append(("k_done", deepcopy(D), deepcopy(P), k, -1, -1, neg))

    steps.append(("done", deepcopy(D), deepcopy(P), -1, -1, -1, None))
    return steps, D, P

def reconstruct(P, src, dst):
    if P[src][dst] is None:
        return None
    path = [dst]
    while path[-1] != src:
        path.append(P[src][path[-1]])
    path.reverse()
    return path

def most_accessible(D):
    best, best_sum = -1, INF
    for v in range(N):
        s = sum(D[v][j] for j in range(N) if D[v][j] < INF and j != v)
        if s < best_sum:
            best_sum, best = s, v
    return best, best_sum

# ───────────────────────── VIZATIMET MJEKESORE ───────────────────────
def draw_hospital(s, x, y, active, tick):
    if active: glow(s, x, y, C["hospital"], 90)
    pygame.draw.rect(s, (160, 30, 45), (x-46, y-42, 92, 74), border_radius=5)
    pygame.draw.rect(s, (230, 70, 90), (x-46, y-42, 92, 74), 2, border_radius=5)
    pygame.draw.rect(s, C["white"], (x-8,  y-32, 16, 54))
    pygame.draw.rect(s, C["white"], (x-28, y-16, 56, 16))
    for ox in (-32, 18):
        for oy2 in (-36, -10):
            pygame.draw.rect(s, (100,180,240), (x+ox, y+oy2, 16, 14))
            pygame.draw.rect(s, (180,220,255), (x+ox, y+oy2, 16, 14), 1)
    pygame.draw.rect(s, (80,15,22), (x-10, y+20, 20, 12))
    a = (tick*0.1) % (2*math.pi)
    sc = int(abs(math.sin(a))*8)
    pygame.draw.circle(s, C["yellow"] if sc>4 else C["red"], (x-28, y-50), 7)
    pygame.draw.circle(s, C["yellow"] if sc<=4 else C["red"], (x+28, y-50), 7)

def draw_ambulance_node(s, x, y, label, active):
    """Nodo ambulance — pa animim te brendshem, ambulanca leviz vetem mbi brinjete."""
    if active: glow(s, x, y, C["ambulance"], 72)
    pygame.draw.rect(s, (170,140,30), (x-34, y-28, 68, 50), border_radius=4)
    pygame.draw.rect(s, (230,200,70), (x-34, y-28, 68, 50), 2, border_radius=4)
    pygame.draw.rect(s, (60,50,10), (x-26, y-18, 52, 40))
    for vi in range(3):
        pygame.draw.line(s, (140,120,30),
                         (x-26, y-18+vi*13), (x+26, y-18+vi*13), 2)
    txt(s, label, 14, C["ambulance"], (x, y+34), center=True, bold=True)

def draw_clinic(s, x, y, label, active):
    if active: glow(s, x, y, C["clinic"], 68)
    pygame.draw.rect(s, (28,120,88), (x-32, y-26, 64, 50), border_radius=4)
    pygame.draw.rect(s, (60,190,145), (x-32, y-26, 64, 50), 2, border_radius=4)
    pygame.draw.rect(s, C["white"], (x-20, y-20, 40, 20), border_radius=3)
    txt(s, label, 13, C["clinic"], (x, y-10), center=True, bold=True)
    pygame.draw.rect(s, (15,65,48), (x-10, y+8, 20, 16))
    for ox in (-24, 14):
        pygame.draw.rect(s, (130,220,200), (x+ox, y-18, 12, 10))

def draw_urgency(s, x, y, label, active, tick):
    if active: glow(s, x, y, C["urgency"], 72)
    pygame.draw.rect(s, (120,48,148), (x-36, y-30, 72, 56), border_radius=5)
    pygame.draw.rect(s, (190,100,220), (x-36, y-30, 72, 56), 2, border_radius=5)
    pygame.draw.rect(s, C["white"], (x-5,  y-22, 10, 34))
    pygame.draw.rect(s, C["white"], (x-16, y-12, 32, 10))
    for ox in (-24, 14):
        col = C["yellow"] if int(tick*0.1+(0 if ox<0 else 1))%2==0 else (80,60,20)
        pygame.draw.rect(s, col, (x+ox, y+2, 12, 14))
    a = (tick*0.12)%(2*math.pi)
    r2 = int(4+abs(math.sin(a))*5)
    pygame.draw.circle(s, C["urgency"], (x-28, y-38), r2)
    pygame.draw.circle(s, C["urgency"], (x+28, y-38), r2)
    txt(s, label, 14, C["purple"], (x, y+40), center=True, bold=True)

# ───────────────────────── BEZIER + RRUGA ────────────────────────────
def bezier_points(p1, p2, ctrl, steps=32):
    pts = []
    for step in range(steps+1):
        t = step / steps
        bx = (1-t)**2*p1[0] + 2*(1-t)*t*ctrl[0] + t**2*p2[0]
        by = (1-t)**2*p1[1] + 2*(1-t)*t*ctrl[1] + t**2*p2[1]
        pts.append((int(bx), int(by)))
    return pts

def get_ctrl_point(p1, p2, u, v):
    """
    Kurbezon brinjete specifike per te shmangur mbivendosjen vizuale.
    - SA (1,2,3) same-column: kthimet kurbohen pak djathtas
    - KR (4,5) same-column: kthimet kurbohen pak majtas
    - QU (6,7) same-column: kthimet kurbohen djathtas
    - Brinjete nderkolumonare qe 'kapercejne': kurbe e mesme
    """
    x1,y1=p1; x2,y2=p2
    mx,my=(x1+x2)/2,(y1+y2)/2

    sa=[1,2,3]; kr=[4,5]; qu=[6,7]

    # SA same-column
    if u in sa and v in sa:
        ui=sa.index(u); vi=sa.index(v)
        if vi < ui:  # kthim prapa (p.sh. 2->1)
            return (mx-42, my), True
        # fqinj normal
        return (mx, my), False

    # KR same-column (vetem 5->4)
    if u in kr and v in kr:
        return (mx+40, my), True

    # QU same-column (vetem 6->7)
    if u in qu and v in qu:
        # kjo eshte drejtvizore normale
        return (mx, my), False

    # Brinja 1->6 (SA1->QU1, kapercen KR kolonan)
    if u==1 and v==6:
        return (mx, my-70), True

    # Brinja 2->5 (SA2->KR-B, kapercen KR-A)
    if u==2 and v==5:
        return (mx+35, my+40), True

    # Brinja 4->7 (KR-A->QU-2, kapercen QU-1)
    if u==4 and v==7:
        return (mx+35, my+50), True

    # Brinja 7->3 (QU-2->SA-3, kthim i gjate)
    if u==7 and v==3:
        return (mx+60, my), True

    # Te gjitha te tjerat: drejtvizore
    return (mx, my), False

def draw_arrow_at(s, tip_pos, prev_pos, color, sz=12):
    tx, ty = tip_pos
    px2, py2 = prev_pos
    a = math.atan2(ty-py2, tx-px2)
    ux, uy = math.cos(a), math.sin(a)
    atip   = (int(tx+ux*sz*0.5), int(ty+uy*sz*0.5))
    aleft  = (int(tx-sz*math.cos(a-0.55)), int(ty-sz*math.sin(a-0.55)))
    aright = (int(tx-sz*math.cos(a+0.55)), int(ty-sz*math.sin(a+0.55)))
    pygame.draw.polygon(s, (0,0,0), [atip,aleft,aright])
    sz2=sz-3
    atip2  = (int(tx+ux*sz2*0.5), int(ty+uy*sz2*0.5))
    aleft2 = (int(tx-sz2*math.cos(a-0.5)), int(ty-sz2*math.sin(a-0.5)))
    aright2= (int(tx-sz2*math.cos(a+0.5)), int(ty-sz2*math.sin(a+0.5)))
    pygame.draw.polygon(s, color, [atip2,aleft2,aright2])

def draw_ambulance_on_road(s, x, y, angle_deg, tick_offset=0):
    """Ambulance e vogel qe leviz mbi brinjen aktive."""
    surf = pygame.Surface((26, 12), pygame.SRCALPHA)
    # Trupi i bardhe
    pygame.draw.rect(surf, C["white"],     (0, 1, 22, 9), border_radius=2)
    pygame.draw.rect(surf, C["ambulance"], (0, 1, 22, 9), 1, border_radius=2)
    # Kryqi i vogel
    pygame.draw.rect(surf, C["red"], (5, 3, 4, 7))
    pygame.draw.rect(surf, C["red"], (3, 5, 8, 3))
    # Kabina
    pygame.draw.rect(surf, C["ambulance"], (22, 0, 4, 11), border_radius=1)
    # Rrota
    for wx in (4, 15):
        pygame.draw.circle(surf, (25,25,25), (wx, 10), 3)
        pygame.draw.circle(surf, (100,100,100), (wx, 10), 2)
    rot = pygame.transform.rotate(surf, angle_deg)
    s.blit(rot, rot.get_rect(center=(x,y)))

def draw_road(s, p1, p2, line_color, arrow_color, width, tick,
              flowing=False, u=-1, v=-1):
    x1,y1=p1; x2,y2=p2
    dx,dy=x2-x1, y2-y1
    ctrl, curved = get_ctrl_point(p1, p2, u, v)

    # Trashesia minimale e brinjeve — gjithmone te dukshme
    draw_w = max(width, 3)

    if curved:
        pts = bezier_points(p1, p2, ctrl, steps=40)
        if len(pts) > 1:
            # Shadow
            pygame.draw.lines(s, (5,8,18),      False, pts, draw_w+8)
            # Brinja kryesore
            pygame.draw.lines(s, line_color,     False, pts, draw_w)
            # Vija e mesit per te dhene thellesi
            pygame.draw.lines(s, (220,235,255),  False, pts, max(1, draw_w//5))

        if flowing and len(pts) > 1:
            t_idx = int((tick*0.008 % 1) * (len(pts)-1))
            kx, ky = pts[min(t_idx, len(pts)-1)]
            if t_idx < len(pts)-1:
                nx2, ny2 = pts[t_idx+1]
                ang = math.degrees(math.atan2(-(ny2-ky), nx2-kx))
            else:
                ang = 0
            draw_ambulance_on_road(s, kx, ky, ang)

        if len(pts) > 2:
            idx  = int(len(pts)*0.68)
            idx2 = min(idx+2, len(pts)-1)
            draw_arrow_at(s, pts[idx2], pts[max(idx-1,0)], arrow_color, sz=14)
    else:
        # Shadow
        pygame.draw.line(s, (5,8,18),     p1, p2, draw_w+8)
        # Brinja kryesore
        pygame.draw.line(s, line_color,   p1, p2, draw_w)
        # Vija e mesit
        pygame.draw.line(s, (220,235,255),p1, p2, max(1, draw_w//5))

        if flowing:
            t  = (tick*0.008)%1
            kx = int(x1+dx*t); ky = int(y1+dy*t)
            ang= math.degrees(math.atan2(-dy, dx))
            draw_ambulance_on_road(s, kx, ky, ang)

        a   = math.atan2(dy, dx)
        tx2, ty2 = x1+dx*0.68, y1+dy*0.68
        sz=14; ux=math.cos(a); uy=math.sin(a)
        atip   = (int(tx2+ux*sz*0.5),  int(ty2+uy*sz*0.5))
        aleft  = (int(tx2-sz*math.cos(a-0.55)), int(ty2-sz*math.sin(a-0.55)))
        aright = (int(tx2-sz*math.cos(a+0.55)), int(ty2-sz*math.sin(a+0.55)))
        pygame.draw.polygon(s, (0,0,0), [atip,aleft,aright])
        sz2=sz-3
        atip2  = (int(tx2+ux*sz2*0.5),  int(ty2+uy*sz2*0.5))
        aleft2 = (int(tx2-sz2*math.cos(a-0.5)), int(ty2-sz2*math.sin(a-0.5)))
        aright2= (int(tx2-sz2*math.cos(a+0.5)), int(ty2-sz2*math.sin(a+0.5)))
        pygame.draw.polygon(s, arrow_color, [atip2,aleft2,aright2])

def edge_style(u, v, k_cur, i_cur, j_cur, phase):
    # Ngjyrat e brinjeve sipas tipit te nyjes source (per dallim me te mire)
    src_type = NTYPE[u]
    idle_col = {
        "hospital":  (140, 40,  55),   # kuq i erret
        "ambulance": (155, 125, 20),   # verdhe i erret
        "clinic":    (28,  120, 88),   # jeshil i erret
        "urgency":   (110, 45,  140),  # vjollce e erret
    }.get(src_type, C["road"])
    idle_arr = {
        "hospital":  (200, 60,  80),
        "ambulance": (210, 170, 40),
        "clinic":    (50,  175, 130),
        "urgency":   (160, 80,  200),
    }.get(src_type, C["arrow_default"])

    if phase in ("init","done","k_done"):
        return idle_col, idle_arr, 5, False

    is_ik = (u==i_cur and v==k_cur)
    is_kj = (u==k_cur and v==j_cur)
    is_ij = (u==i_cur and v==j_cur)

    if phase == "update":
        if is_ij: return C["green"],  C["arrow_update"], 8, False
        if is_ik: return C["cyan"],   C["arrow_ik"],     7, True
        if is_kj: return C["orange"], C["arrow_kj"],     7, True

    if phase == "check":
        if is_ik: return C["yellow"], C["arrow_check"], 6, True
        if is_kj: return C["yellow"], C["arrow_check"], 6, True
        if is_ij: return (85,105,140), C["arrow_default"], 5, False

    if phase == "check_inf":
        if is_ik: return (55,70,100), C["arrow_inf"], 4, False
        if is_kj: return (55,70,100), C["arrow_inf"], 4, False

    return idle_col, idle_arr, 5, False

# ───────────────────────── MATRICA ───────────────────────────────────
def draw_matrix(s, D, gw, H, k, i, j, phase, query_path=None):
    px = gw
    pw = s.get_width() - gw
    pad = 12

    txt(s, "Matrica  D[i][j]  (minuta)", 14, C["white"],
        (px+pw//2, 16), center=True, bold=True)

    # Header labels kompakt
    hlabels = ["SQ","S1","S2","S3","KA","KB","U1","U2"]
    cell = min(36, (pw - pad*2) // (N+1))
    total_w = cell*(N+1)
    ox = px + (pw-total_w)//2
    oy = 36

    for ci in range(N):
        cx=ox+(ci+1)*cell
        r=pygame.Rect(cx+1, oy+1, cell-2, cell-2)
        pygame.draw.rect(s, (18,32,65), r, border_radius=3)
        pygame.draw.rect(s, C["cyan"], r, 1, border_radius=3)
        txt(s, hlabels[ci], 9, C["cyan"], (cx+cell//2, oy+cell//2), center=True, bold=True)

    for ri in range(N):
        ry2=oy+(ri+1)*cell
        r=pygame.Rect(ox+1, ry2+1, cell-2, cell-2)
        pygame.draw.rect(s, (18,32,65), r, border_radius=3)
        pygame.draw.rect(s, C["cyan"], r, 1, border_radius=3)
        txt(s, hlabels[ri], 9, C["cyan"], (ox+cell//2, ry2+cell//2), center=True, bold=True)

    for ri in range(N):
        for rj in range(N):
            cx2=ox+(rj+1)*cell
            cy2=oy+(ri+1)*cell
            r=pygame.Rect(cx2+1, cy2+1, cell-2, cell-2)

            if phase=="update" and ri==i and rj==j:
                bg,fc,brd=(15,65,25),  C["green"],    C["green"]
            elif phase in ("update","check") and ((ri==i and rj==k) or (ri==k and rj==j)):
                bg,fc,brd=(14,45,70),  C["cyan"],     C["cyan"]
            elif phase=="check_inf" and ((ri==i and rj==k) or (ri==k and rj==j)):
                bg,fc,brd=(20,26,40),  (75,95,125),   (45,60,85)
            elif ri==k or rj==k:
                bg,fc,brd=(20,33,58),  (120,148,192), (35,52,82)
            elif ri==rj:
                bg,fc,brd=(10,16,32),  C["dim"],      (28,40,62)
            else:
                bg,fc,brd=(10,15,30),  C["dim"],      (28,38,60)

            if query_path and len(query_path)>1:
                for qi in range(len(query_path)-1):
                    if ri==query_path[qi] and rj==query_path[qi+1]:
                        bg,fc,brd=(48,15,70),C["purple"],C["purple"]

            pygame.draw.rect(s, bg,  r, border_radius=3)
            pygame.draw.rect(s, brd, r, 1, border_radius=3)
            val=D[ri][rj]
            vstr="inf" if val==INF else str(int(val))
            txt(s, vstr, 11, fc, (cx2+cell//2, cy2+cell//2),
                center=True, bold=(val<INF))

    matrix_bottom = oy+(N+1)*cell

    # Legjenda
    ly = matrix_bottom+8
    pygame.draw.line(s, (35,50,80), (px+pad,ly), (px+pw-pad,ly))
    ly+=7
    txt(s, "LEGJENDA", 11, C["dim"], (px+pw//2,ly+1), center=True, bold=True)
    ly+=15

    leg_items = [
        (C["green"],   (15,65,25),  "UPDATE — rruge me e shkurter gjetur"),
        (C["cyan"],    (14,45,70),  "i->k dhe k->j (segmentet aktive)"),
        (C["yellow"],  (45,40,10),  "CHECK — pa permiresim"),
        ((75,95,125),  (20,26,40),  "INF — nuk ka rruge nepermjet k"),
        (C["purple"],  (48,15,70),  "QUERY — rruga e kerkuar"),
    ]
    for brd_c, bg_c, label in leg_items:
        item_r=pygame.Rect(px+pad, ly, pw-pad*2, 19)
        pygame.draw.rect(s, bg_c,  item_r, border_radius=4)
        pygame.draw.rect(s, brd_c, item_r, 1, border_radius=4)
        pygame.draw.rect(s, brd_c, (px+pad+4, ly+4, 11, 11), border_radius=2)
        txt(s, label, 10, (210,225,245), (px+pad+20, ly+4))
        ly+=22

    return ly

# ───────────────────────── APLIKACIONI ───────────────────────────────
class App:
    def __init__(self):
        self.windowed = (1440, 860)
        self.full = False
        self.screen = pygame.display.set_mode(self.windowed, pygame.RESIZABLE)
        pygame.display.set_caption("Floyd-Warshall — Rrjeti Mjekesor i Urgjences")
        self.clock = pygame.time.Clock()
        self.tick = 0
        self.paused = True
        self.counter = 0
        self.speed = 2
        self.query_src = 0
        self.query_dst = 7
        self.show_query = False
        self.reset()

    def reset(self):
        self.steps, self.final_D, self.final_P = floyd_warshall_steps()
        self.step_idx = 0
        self.cur = self.steps[0]
        self.logs = []
        self.log("=== Rrjeti Mjekesor — Floyd-Warshall ===", C["blue"])
        self.log(f"Rrjet: {N} nyje, {len(EDGES_RAW)} rruge te drejtuara.", C["dim"])
        self.log("Qellimi: gjej kohen minimale ambulances", C["dim"])
        self.log("ndermjet CDO cift nyjesh ne rrjet.", C["dim"])
        self.log("SPACE: luaj/pauzo | S: hap | G: query", C["dim"])

    def log(self, msg, col):
        self.logs.append((msg, col))
        self.logs = self.logs[-25:]

    def node_pos(self):
        W, H = self.graph_w, self.screen.get_height()
        top=95; bot=H-65; span=bot-top
        return {
            0: (int(W*0.09), int(top+span*0.50)),  # SQ - majtas qender
            1: (int(W*0.32), int(top+span*0.12)),  # SA-1 - lart
            2: (int(W*0.32), int(top+span*0.45)),  # SA-2 - qender
            3: (int(W*0.32), int(top+span*0.82)),  # SA-3 - posht
            4: (int(W*0.62), int(top+span*0.18)),  # KR-A - lart
            5: (int(W*0.62), int(top+span*0.65)),  # KR-B - posht
            6: (int(W*0.87), int(top+span*0.32)),  # QU-1 - lart
            7: (int(W*0.87), int(top+span*0.75)),  # QU-2 - posht
        }

    def apply_step(self):
        if self.step_idx >= len(self.steps)-1:
            return
        self.step_idx += 1
        self.cur = self.steps[self.step_idx]
        kind, D, P, k, i, j, extra = self.cur

        def nm(idx): return LABELS[idx]

        if kind == "update":
            old, new = extra
            old_s = "INF" if old==INF else f"{int(old)}min"
            self.log(
                f"UPDATE D[{SHORT[i]}][{SHORT[j]}]: {old_s}->{int(new)}min",
                C["green"])
            self.log(
                f"  {nm(i)}->{nm(k)}->{nm(j)} eshte rruge me e shkurter!",
                C["cyan"])

        elif kind == "check":
            via = D[i][k]+D[k][j]
            cur = D[i][j]
            cur_s = "INF" if cur==INF else f"{int(cur)}min"
            self.log(
                f"CHECK [{SHORT[i]}]->[{SHORT[j]}] via {nm(k)}: "
                f"{int(via)}min >= {cur_s} — pa ndryshim",
                C["yellow"])

        elif kind == "check_inf":
            ik_inf = D[i][k]==INF
            kj_inf = D[k][j]==INF
            if ik_inf and kj_inf:
                reason = f"{nm(i)}->{nm(k)} dhe {nm(k)}->{nm(j)}=INF"
            elif ik_inf:
                reason = f"{nm(i)}->{nm(k)}=INF"
            else:
                reason = f"{nm(k)}->{nm(j)}=INF"
            self.log(
                f"INF [{SHORT[i]}]->[{SHORT[j]}] via {nm(k)}: {reason}",
                (70,85,110))

        elif kind == "k_done":
            if extra:
                self.log(f"GABIM: Cikel negativ pas k={nm(k)}!", C["red"])
            else:
                upd_k = sum(1 for st in self.steps[:self.step_idx+1]
                           if st[0]=="update" and st[3]==k)
                self.log(f"-- k={SHORT[k]} ({nm(k)}) kompletuar --", C["active_k"])
                if upd_k > 0:
                    self.log(
                        f"   {upd_k} rruge te optimizuara via {nm(k)}.",
                        C["green"])
                else:
                    self.log(
                        f"   {nm(k)} nuk ofroi asnje shkurtim.",
                        C["dim"])

        elif kind == "done":
            total_upd = sum(1 for st in self.steps if st[0]=="update")
            reachable = sum(1 for r in range(N) for c in range(N)
                           if r!=c and self.final_D[r][c]<INF)
            best_v, best_sum = most_accessible(D)
            # Rruga me e shpejte per ambulance
            best_resp = min(
                (D[r][c],r,c) for r in range(N) for c in range(N)
                if r!=c and D[r][c]<INF
            )
            worst_resp = max(
                (D[r][c],r,c) for r in range(N) for c in range(N)
                if r!=c and D[r][c]<INF
            )
            self.log("=== FLOYD-WARSHALL PERFUNDOI ===", C["green"])
            self.log(f"Optimizime totale: {total_upd} rruge te permiresura.", C["green"])
            self.log(f"Cefte te arritshme: {reachable} nga {N*(N-1)}.", C["cyan"])
            self.log(
                f"Pergjigja me e shpejte: {LABELS[best_resp[1]]}->{LABELS[best_resp[2]]}"
                f" = {int(best_resp[0])}min", C["cyan"])
            self.log(
                f"Pergjigja me e ngadalt: {LABELS[worst_resp[1]]}->{LABELS[worst_resp[2]]}"
                f" = {int(worst_resp[0])}min", C["dim"])
            self.log(
                f"Qendra me e aksesueshme: {LABELS[best_v]} (Sd={int(best_sum)}min)",
                C["blue"])
            self.log("Algoritmi vleu: tani dime kohen e sakte", C["blue"])
            self.log("te ambulances per CDO destinacion!", C["blue"])
            self.paused = True

    def _status_color(self):
        kind = self.cur[0]
        return {"update":C["green"],"check":C["yellow"],
                "check_inf":(60,75,110),"k_done":C["active_k"],
                "done":C["green"],"init":C["blue"]}.get(kind, C["dim"])

    def _status_short(self):
        kind,D,P,k,i,j,extra = self.cur
        if kind=="init":    return "Inicializim — matrica me kohet fillestare"
        if kind=="done":
            best,bs = most_accessible(D)
            return f"Perfundoi! Qendra me aksesueshme: {LABELS[best]} (Sd={int(bs)}min)"
        if kind=="k_done":  return f"k={SHORT[k]} ({LABELS[k]}) kompletuar"
        if kind=="update":
            old,new=extra
            os2="INF" if old==INF else f"{int(old)}min"
            return f"D[{SHORT[i]}][{SHORT[j]}]: {os2}->{int(new)}min  via {LABELS[k]}"
        if kind=="check":
            via=D[i][k]+D[k][j]; cur=D[i][j]
            cs2="INF" if cur==INF else f"{int(cur)}min"
            return f"D[{SHORT[i]}][{SHORT[j]}]: {int(via)}min >= {cs2}  no update"
        if kind=="check_inf":
            return f"D[{SHORT[i]}][{SHORT[j]}] via {LABELS[k]}: segment = INF"
        return ""

    def draw_graph(self):
        s = self.screen
        W, H = self.graph_w, s.get_height()
        pygame.draw.rect(s, C["graph"], (12,80,W-24,H-130), border_radius=16)

        pos = self.node_pos()
        kind,D,P,k,i,j,extra = self.cur

        # Harqet
        for u,v,w in EDGES_RAW:
            lc,ac,width,flowing = edge_style(u,v,k,i,j,kind)
            draw_road(s, pos[u], pos[v], lc, ac, width, self.tick,
                      flowing=flowing, u=u, v=v)

        # Rruga query
        qp = None
        if self.show_query:
            qp = reconstruct(self.final_P, self.query_src, self.query_dst)
            if qp and len(qp)>1:
                for qi in range(len(qp)-1):
                    pygame.draw.line(s, C["purple"],
                                     pos[qp[qi]], pos[qp[qi+1]], 6)

        # Peshat e harqeve — etiketë e qarte dhe e lidhur me brinjen
        # Secila brinjë ka ngjyrën e tipit të nyjes burimore
        edge_set_uv = {(u2,v2) for u2,v2,_ in EDGES_RAW}
        bidir_pairs = set()
        for u2,v2,_ in EDGES_RAW:
            if (v2,u2) in edge_set_uv:
                bidir_pairs.add((min(u2,v2), max(u2,v2)))

        # Ngjyrat e etiketave sipas tipit te nyjes source
        type_label_col = {
            "hospital":  (255, 100, 115),  # kuq i ndritshëm
            "ambulance": (255, 220,  70),  # verdhe i ndritshëm
            "clinic":    ( 60, 210, 155),  # jeshil mjekesor
            "urgency":   (195, 110, 255),  # vjollce e ndritshme
        }
        type_border_col = {
            "hospital":  (180,  45,  60),
            "ambulance": (180, 145,  20),
            "clinic":    ( 25,  95,  68),
            "urgency":   (110,  40, 155),
        }

        for u,v,w in EDGES_RAW:
            x1,y1=pos[u]; x2,y2=pos[v]
            ctrl, curved = get_ctrl_point(pos[u], pos[v], u, v)
            is_bidir = (min(u,v), max(u,v)) in bidir_pairs
            src_type = NTYPE[u]
            lbl_col = type_label_col.get(src_type, C["white"])
            brd_col = type_border_col.get(src_type, C["road_hi"])

            if curved:
                # Midpoint i kurbës Bezier (t=0.5)
                cx3 = int(0.25*x1 + 0.5*ctrl[0] + 0.25*x2)
                cy3 = int(0.25*y1 + 0.5*ctrl[1] + 0.25*y2)
                # Drejtim normal nga qendra e kurbës — offset i madh
                cdx = cx3 - (x1+x2)//2; cdy = cy3 - (y1+y2)//2
                cL  = math.hypot(cdx,cdy) or 1
                # Etiketa shkon ne anen e kurbës (anon nga brinja)
                off = 32
                lx  = cx3 + int(cdx/cL * off)
                ly  = cy3 + int(cdy/cL * off)
                # Linja lidhëse nga brinja deri etiketa
                mid_on_curve = (cx3, cy3)
            else:
                dx2,dy2 = x2-x1, y2-y1
                L2 = math.hypot(dx2,dy2) or 1
                # Bidireksionale: pozicione te ndryshme (35%/65%)
                t_pos = 0.35 if u < v else 0.65
                bx = int(x1 + dx2*t_pos); by = int(y1 + dy2*t_pos)
                # Normal perpendikular — ane te ndryshme
                side  = 1 if u < v else -1
                nx_   = -dy2/L2 * side
                ny_   =  dx2/L2 * side
                # Offset: 38px bidireksionale, 28px njekaheshe
                off   = 38 if is_bidir else 28
                lx    = int(bx + nx_*off)
                ly    = int(by + ny_*off)
                mid_on_curve = (bx, by)

            # Linja e hollë lidhëse nga brinja deri te etiketa
            pygame.draw.line(s, brd_col, mid_on_curve, (lx, ly), 1)
            # Pika e lidhjes mbi brinjen
            pygame.draw.circle(s, brd_col, mid_on_curve, 3)

            # Kutia e etiketës
            label_str = f"{w}min"
            lbl_surf_w = fnt(12, bold=True).size(label_str)[0]
            box_w = lbl_surf_w + 16
            box_h = 22
            r2 = pygame.Rect(lx - box_w//2, ly - box_h//2, box_w, box_h)
            # Sfond i plotë opak
            pygame.draw.rect(s, (3, 5, 15), r2, border_radius=5)
            # Border me ngjyrën e tipit
            pygame.draw.rect(s, brd_col, r2, 2, border_radius=5)
            # Tekst i ndritshëm dhe bold
            txt(s, label_str, 12, lbl_col, r2.center, center=True, bold=True)


        # Nyjet
        for ni in range(N):
            x,y=pos[ni]
            is_k  = (ni==k and kind not in ("init","done"))
            is_ij = (ni in (i,j) and kind in ("update","check","check_inf"))
            active = is_k or is_ij or (qp and ni in qp)
            tp = NTYPE[ni]

            if tp=="hospital":
                draw_hospital(s, x, y, active, self.tick)
                txt(s, "SPITALI QENDROR", 14, C["hospital"], (x,y+90), center=True, bold=True)
                txt(s, SHORT[0], 11, C["dim"], (x,y+106), center=True)
            elif tp=="ambulance":
                draw_ambulance_node(s, x, y, LABELS[ni], active)
                txt(s, "Stacion Amb.", 10, C["dim"], (x,y+50), center=True)
            elif tp=="clinic":
                draw_clinic(s, x, y, LABELS[ni], active)
                txt(s, "Klinike Rajonale", 10, C["dim"], (x,y+44), center=True)
            else:
                draw_urgency(s, x, y, LABELS[ni], active, self.tick)
                txt(s, "Q. Urgjence", 10, C["dim"], (x,y+56), center=True)

        # Etiketa k
        if k>=0 and kind not in ("done",):
            kx,ky=pos[k]
            r=pygame.Rect(kx-42, ky-95, 84, 26)
            pygame.draw.rect(s, (50,40,8), r, border_radius=6)
            pygame.draw.rect(s, C["active_k"], r, 2, border_radius=6)
            txt(s, f"k={SHORT[k]} ({LABELS[k]})", 13, C["active_k"],
                r.center, center=True, bold=True)

        # Titull
        pygame.draw.rect(s, (6,10,24), (12,4,W-24,76), border_radius=10)
        txt(s, "FLOYD-WARSHALL — Rrjeti Mjekesor i Urgjences",
            21, C["white"], (W//2,24), center=True, bold=True)
        txt(s, "Spital Qendror  ->  Stacione Ambulance  ->  Klinika  ->  Qendra Urgjence",
            12, C["blue"], (W//2,48), center=True)

        # Bara e statusit
        bar_col = {
            "update":(12,55,18),"check":(50,42,8),
            "check_inf":(12,16,36),"k_done":(48,40,8),
            "done":(10,52,18),"init":(10,20,50),
        }.get(kind, (8,12,28))
        pygame.draw.rect(s, bar_col, (12,H-46,W-24,36), border_radius=8)
        pygame.draw.rect(s, self._status_color(), (12,H-46,W-24,36), 1, border_radius=8)
        txt(s, self._status_short(), 13, C["white"], (W//2,H-28), center=True)

    def draw_panel(self):
        s = self.screen
        W_full = s.get_width()
        H = s.get_height()
        px = self.graph_w
        pw = W_full - px
        pad = 10

        pygame.draw.rect(s, (8,13,28),  (px,0,pw,H))
        pygame.draw.rect(s, (14,22,46), (px+6,6,pw-12,H-12), border_radius=12)

        kind,D,P,k,i,j,extra = self.cur

        # Zona 1: Matrica + Legjenda
        qp = reconstruct(self.final_P, self.query_src, self.query_dst) \
             if self.show_query else None
        leg_bottom = draw_matrix(s, D, px, H, k, i, j, kind, qp)

        # Zona 2: Status aktual
        info_y = leg_bottom+6
        status_txt = self._status_short()
        info_r = pygame.Rect(px+pad, info_y, pw-pad*2, 44)
        pygame.draw.rect(s, (12,20,48), info_r, border_radius=6)
        pygame.draw.rect(s, self._status_color(), info_r, 2, border_radius=6)

        kind_label = {"init":"INIT","update":"UPDATE","check":"CHECK",
                      "check_inf":"INF","k_done":"K DONE","done":"DONE"}.get(kind, kind.upper())
        badge_c = self._status_color()
        badge_r = pygame.Rect(px+pad+6, info_y+6, 58, 16)
        pygame.draw.rect(s, badge_c, badge_r, border_radius=3)
        txt(s, kind_label, 10, (5,8,18), badge_r.center, center=True, bold=True)
        txt(s, status_txt, 11, (220,235,255), (px+pad+8, info_y+26))

        total=len(self.steps)
        txt(s, f"{self.step_idx+1}/{total}", 11, C["dim"],
            (px+pw-pad-8, info_y+6))

        info_bottom = info_y+52

        # Zona 3: Log
        log_y = info_bottom+5
        pygame.draw.line(s, (30,45,75), (px+pad,log_y), (px+pw-pad,log_y))
        log_y+=5
        txt(s, "LOG EKZEKUTIMI", 11, C["dim"], (px+pad+4,log_y), bold=True)
        log_y+=14

        max_chars = max(24, (pw-pad*2-8)//7)
        for msg,col in self.logs[-11:]:
            log_r=pygame.Rect(px+pad, log_y, pw-pad*2, 14)
            pygame.draw.rect(s, (10,16,34), log_r)
            display = msg if len(msg)<=max_chars else msg[:max_chars-2]+".."
            txt(s, display, 10, col, (px+pad+4, log_y+1))
            log_y+=15
            if log_y > H-115:
                break

        # Zona 4: Query
        ctrl_y = H-108
        pygame.draw.line(s, (30,45,75), (px+pad,ctrl_y), (px+pw-pad,ctrl_y))
        ctrl_y+=6

        src_lbl=LABELS[self.query_src]; dst_lbl=LABELS[self.query_dst]
        q_r=pygame.Rect(px+pad, ctrl_y, pw-pad*2, 32)
        pygame.draw.rect(s, (14,18,40), q_r, border_radius=5)
        pygame.draw.rect(s, C["purple"], q_r, 1, border_radius=5)
        txt(s, f"QUERY  {src_lbl} -> {dst_lbl}", 11, C["purple"],
            (px+pad+8, ctrl_y+4), bold=True)

        if self.show_query:
            qp2=reconstruct(self.final_P, self.query_src, self.query_dst)
            if kind=="done" and qp2:
                d=self.final_D[self.query_src][self.query_dst]
                path_str=" -> ".join(LABELS[n] for n in qp2)
                txt(s, f"{path_str}  =  {int(d)}min", 11, C["green"],
                    (px+pad+8, ctrl_y+18), bold=True)
            elif kind!="done":
                txt(s, "shtyp G pas perfundimit", 10, C["dim"],
                    (px+pad+8, ctrl_y+18))
        else:
            txt(s, "G: aktivizo / fsheh rrugen", 10, C["dim"],
                (px+pad+8, ctrl_y+18))

        ctrl_y+=38
        for ln in [
            "SPACE: luaj/pauzo    S/->: hap    R: rinicio",
            "Q/W: burimi    E/D: dest.    G: query    F: fullscreen",
        ]:
            txt(s, ln, 10, (80,100,140), (px+pad+4, ctrl_y))
            ctrl_y+=14

    def draw(self):
        self.screen.fill(C["bg"])
        self.draw_graph()
        self.draw_panel()
        pygame.display.flip()

    def resize(self, w, h):
        self.graph_w = int(w*0.62)

    def toggle_full(self):
        self.full = not self.full
        if self.full:
            self.screen = pygame.display.set_mode((0,0), pygame.FULLSCREEN)
        else:
            self.screen = pygame.display.set_mode(self.windowed, pygame.RESIZABLE)
        self.resize(*self.screen.get_size())

    def events(self):
        for e in pygame.event.get():
            if e.type==pygame.QUIT: return False
            if e.type==pygame.VIDEORESIZE and not self.full:
                self.windowed=(e.w,e.h)
                self.screen=pygame.display.set_mode(self.windowed, pygame.RESIZABLE)
                self.resize(e.w,e.h)
            if e.type==pygame.KEYDOWN:
                if e.key==pygame.K_ESCAPE: return False
                elif e.key==pygame.K_SPACE: self.paused=not self.paused
                elif e.key in (pygame.K_s,pygame.K_RIGHT): self.apply_step()
                elif e.key==pygame.K_r: self.reset()
                elif e.key in (pygame.K_f,pygame.K_F11): self.toggle_full()
                elif e.key==pygame.K_q:
                    self.query_src=(self.query_src-1)%N
                elif e.key==pygame.K_w:
                    self.query_src=(self.query_src+1)%N
                elif e.key==pygame.K_e:
                    self.query_dst=(self.query_dst-1)%N
                elif e.key==pygame.K_d:
                    self.query_dst=(self.query_dst+1)%N
                elif e.key==pygame.K_g:
                    self.show_query=not self.show_query
        return True

    def run(self):
        self.resize(*self.windowed)
        running=True
        while running:
            self.tick+=1
            running=self.events()
            if not self.paused:
                self.counter+=1
                if self.counter>=max(1,60//self.speed):
                    self.apply_step()
                    self.counter=0
            self.draw()
            self.clock.tick(60)
        pygame.quit()
        sys.exit()

# ───────────────────────── STATISTIKAT FINALE ────────────────────────
def print_results(D, P):
    print("\n"+"="*70)
    print("  FLOYD-WARSHALL — REZULTATET FINALE (Rrjeti Mjekesor i Urgjences)")
    print("="*70)

    for v in range(N):
        if D[v][v]<0:
            print(f"\n  GABIM KRITIK: Cikel negativ te nyja {LABELS[v]}!")
            return

    print("\n  Matrica e Distancave Minimale D[i][j] (minuta):\n")
    header="           "+"  ".join(f"{s:>6}" for s in LABELS)
    print(header)
    print("           "+"-"*(len(header)-11))
    for ri in range(N):
        row=f"  {LABELS[ri]:<8} |"
        for rj in range(N):
            val=D[ri][rj]
            row+=f"  {'INF':>6}" if val==INF else f"  {int(val):>6}"
        print(row)

    print(f"\n  Kompleksiteti: O(V^3) = O({N}^3) = O({N**3}) operacione")
    print(f"  E pranueshme per V<=200; per rrjete te medha prefer Dijkstra per cdo burim.")

    print("\n  Aksesueshmeria e Qendrave (shuma e distancave):\n")
    totals=[]
    for v in range(N):
        total=sum(D[v][j] for j in range(N) if D[v][j]<INF and j!=v)
        reach=sum(1 for j in range(N) if D[v][j]<INF and j!=v)
        totals.append((total,v,reach))
    totals.sort()
    for total,v,reach in totals:
        star=" <-- ME E AKSESUESHME" if v==totals[0][1] else ""
        print(f"    {LABELS[v]:<8}: Sd={int(total):>4}min  (arrin {reach}/{N-1} nyje){star}")

    print("\n  Rruget Optimale nga Spitali Qendror:\n")
    for dst in range(1,N):
        path=reconstruct(P,0,dst)
        if path:
            ps=" -> ".join(LABELS[n] for n in path)
            print(f"    SQ -> {LABELS[dst]:<8}: {int(D[0][dst]):>3}min  |  {ps}")
        else:
            print(f"    SQ -> {LABELS[dst]:<8}: E PAARRITSHME")

    print("\n"+"="*70)
    print("  PERFUNDIM: CFARE NA VLEU ALGORITMI?")
    print("="*70)

    best_v,best_sum=most_accessible(D)
    all_d=[(D[r][c],r,c) for r in range(N) for c in range(N)
           if r!=c and D[r][c]<INF]
    best_resp=min(all_d); worst_resp=max(all_d)
    avg_d=sum(x[0] for x in all_d)/len(all_d)

    reachable=len(all_d)
    unreachable=N*(N-1)-reachable

    # Count updates
    D2,P2=build_matrix()
    upd_count=0
    for k in range(N):
        for ii in range(N):
            for jj in range(N):
                if D2[ii][k]<INF and D2[k][jj]<INF:
                    nv=D2[ii][k]+D2[k][jj]
                    if nv<D2[ii][jj]:
                        D2[ii][jj]=nv; P2[ii][jj]=P2[k][jj]
                        upd_count+=1

    print(f"\n  Optimizime te kryera: {upd_count} rruge te permiresura.")
    print(f"  Cefte te arritshme: {reachable} nga {N*(N-1)} te mundshme.")
    if unreachable:
        print(f"  Cefte te PA-arritshme: {unreachable} (graf jo i plote / njekaheshe).")
    print(f"\n  Koha me e shkurter pergjigje: "
          f"{LABELS[best_resp[1]]}->{LABELS[best_resp[2]]} = {int(best_resp[0])}min")
    print(f"  Koha me e gjate pergjigje:   "
          f"{LABELS[worst_resp[1]]}->{LABELS[worst_resp[2]]} = {int(worst_resp[0])}min")
    print(f"  Koha mesatare: {avg_d:.1f}min")
    print(f"\n  Qendra me e aksesueshme: {LABELS[best_v]} (Sd={int(best_sum)}min)")
    print(f"  => Vendosja e ambulances ne {LABELS[best_v]} minimizon kohen mesatare")
    print(f"     te pergjigjes ndaj te gjitha nyjeve te tjera ne rrjet.")

    print("\n  Vlera praktike per sistemin mjekesor:")
    print("    a) Dispeceri di kohen EKZAKTE per cdo destinacion.")
    print("    b) Ambulanca ndiqet per rruge OPTIMALE — kursehet koha kritike.")
    print("    c) Nese nje rruge bllokohet, sistemi mund te rillogarise shpejt.")
    print("    d) Mund te identifikohen 'nyjet kritike' pa te cilat rrjeti prishet.")
    print("="*70+"\n")


if __name__=="__main__":
    _, D, P = floyd_warshall_steps()
    print_results(D, P)
    App().run()


  FLOYD-WARSHALL — REZULTATET FINALE (Rrjeti Mjekesor i Urgjences)

  Matrica e Distancave Minimale D[i][j] (minuta):

               SQ    SA-1    SA-2    SA-3    KR-A    KR-B    QU-1    QU-2
           --------------------------------------------------------------
  SQ       |       0       8      12      15      18      23      25      28
  SA-1     |       8       0      20      23      10      31      17      23
  SA-2     |      12       6       0      27      14      11      21      19
  SA-3     |      49      41      61       0      14       9      21      13
  KR-A     |      35      27      47      30       0      39       7      13
  KR-B     |      40      32      52      25       5       0      12       8
  QU-1     |      28      20      40      23      30      32       0       6
  QU-2     |      66      58      78      17      31      26      38       0

  Kompleksiteti: O(V^3) = O(8^3) = O(512) operacione
  E pranueshme per V<=200; per rrjete te medha prefer Dijkstra

SystemExit: 

---

## 3 - Edmonds-Karp: Rrjedhat Maksimale

### Çfarë problemi zgjidhim?

Imagjinoni **rrjetin e ujësjellësit urban** ku:
- Një rezervuar (**burim**) duhet të furnizojë lagjet  
- Çdo tub ka **kapacitet maksimal** të rrjedhës  
- Duam të gjejmë **sa ujë maksimal** mund të kalojë nga rezervuari te destinacioni

### Konceptet Themelore

| Koncept | Përkufizimi | Shembull |
|---------|-------------|---------|
| **Rrjedhë** f(u,v) | Uji që kalon nëpër tub | 8 njësi/sek |
| **Kapacitet** c(u,v) | Kapaciteti max i tubit | 15 njësi/sek |
| **Graf Mbetes** | Grafiku i kapaciteteve të lira | c(u,v) - f(u,v) |
| **Rrugë Rritëse** | Rrugë nga s tek t në grafun mbetes | s→A→B→t |
| **Ngushtica** | Kapaciteti minimal në rrugë | min(7,12,4) = 4 |

### Kushtëzimi i Rrjedhës

1. **Kapaciteti:** `0 ≤ f(u,v) ≤ c(u,v)` — nuk mund të kalosh kapacitetin
2. **Konservimi:** Rrjedha që hyn = Rrjedha që del (çdo nyje përveç s dhe t)
3. **Antisimetria:** `f(u,v) = -f(v,u)`

### Algoritmi Edmonds-Karp

**Ideja:** Gjej rrugë rritëse me BFS (garanton rrugën më të shkurtër), përditëso rrjedhën.

```
ndërsa ekziston rrugë rritëse P nga s tek t (via BFS):
    b = min { c(u,v) - f(u,v) : (u,v) ∈ P }   ← ngushtica
    për çdo hark (u,v) ∈ P:
        f(u,v) += b    # shto rrjedhën
        f(v,u) -= b    # hark i kthyer
rrjedha_max = Σ f(s,v)


---

## Ushtrimi 3 — Menaxhimi i Rrjetit Urban të Ujësjellësit gjatë Emergjencës

---------------------------------------------------------------------------------------------------------------------------------------------

### Skenari Real

Qyteti po përballet me një situatë emergjente pas një defekti të madh në rrjetin kryesor të ujësjellësit.
Rezervuari qendror i qytetit duhet të furnizojë me ujë disa lagje përmes stacioneve të pompimit.

Për shkak të kufizimeve fizike të tubacioneve:

çdo tub ka një kapacitet maksimal të ujit që mund të transportojë;
stacionet e pompimit kanë kufizime operative;
disa lagje janë të lidhura edhe ndërmjet tyre për të ndihmuar shpërndarjen e ujit;
objektivi kryesor është që sa më shumë ujë të arrijë te Qendra Emergjente e Qytetit.

Administrata e ujësjellësit dëshiron të dijë:

“Cila është sasia maksimale e ujit që mund të transportohet nga rezervuari deri te qendra emergjente pa tejkaluar kapacitetet e tubacioneve?”

---------------------------------------------------------------------------------------------------------------------------------------------

### Objektivi i Problemit

Të gjendet Rrjedha Maksimale pra: sasia më e madhe e ujit që mund të dërgohet nga burimi deri te derdhja.

---------------------------------------------------------------------------------------------------------------------------------------------

### Pse përdoret Edmonds-Karp?

#### Në rrjete reale:

nuk mund të dërgojmë ujë pafundësisht;
çdo tub ka kufizim;
rrjedha duhet të respektojë kapacitetet;
mund të ekzistojnë shumë rrugë alternative.

#### Algoritmi Edmonds-Karp:

gjen automatikisht rrugët më të mira;
përdor BFS për të kërkuar rrugë rritëse;
përditëson rrjedhën hap pas hapi;
garanton gjetjen e rrjedhës maksimale.

---------------------------------------------------------------------------------------------------------------------------------------------

### Struktura e Rrjetit

```
REZERVUARI ──→ [SP-A, SP-B, SP-C] ──→ [L-1, L-2, L-3, L-4] ──→ QE
  (Burim)       (Stacionet Pompimit)      (Lagjet)           (Derdhja)
```


### Implementimi i Edmonds-Karp

In [1]:
"""
Ushtrimi 3 - Edmonds-Karp: Rrjeti Urban i Ujesjellesit gjate Emergjences
Rezervuari -> Stacione Pompimi -> Lagje -> Qendra Emergjente

RREGULLIMET KRYESORE:
- Rrjedhezat fillojne te gjitha 0/cap — nuk ekzekutohet algoritmi perpara vizualizimit
- Cdo hap ruan snapshot-in e rrjedhezave (flow_snap) — etiketa tregon gjendjen AKTUALE
- BFS, augment, update jane hapa te ndare te dukshem
- Rruga rritese dhe ngushtica shfaqen qarte para se rrjedha te ndryshohet
"""

import pygame, sys, math
from collections import deque, defaultdict
from copy import deepcopy

pygame.init()

FONTS = {}
C = {
    "bg":        (8,  14,  28),
    "graph":     (10, 18,  35),
    "panel":     (15, 24,  46),
    "panel2":    (20, 32,  60),
    "white":     (230,245, 255),
    "dim":       (125,148, 190),
    "blue":      (70, 190, 255),
    "cyan":      (40, 220, 255),
    "yellow":    (240,190, 40),
    "red":       (255,80,  55),
    "green":     (60, 220, 120),
    "purple":    (155,120, 255),
    "orange":    (255,160, 60),
    "pipe":      (55, 95,  120),
    "pipe_hi":   (85, 140, 170),
    "pipe_dark": (18, 30,  48),
    "water":     (45, 190, 245),
    "active_k":  (255,210, 55),
}

# ─── GRAFIKU ────────────────────────────────────────────────────────────────
# (burim, destinacion, kapacitet)
EDGES = [
    (0,1,20),(0,2,15),(0,3,12),
    (1,4,14),(1,5,10),
    (2,4,8), (2,5,12),(2,6,7),
    (3,5,9), (3,6,11),(3,7,6),
    (4,5,5), (5,6,4), (6,7,6),
    (4,8,15),(5,8,13),(6,8,11),(7,8,9),
]

NODE_NAMES = {
    0:"Rezervuari", 1:"SP-A", 2:"SP-B", 3:"SP-C",
    4:"L-1", 5:"L-2", 6:"L-3", 7:"L-4", 8:"QE"
}
N_NODES = 9
SOURCE, SINK = 0, 8

# ─── FONT / TEXT ────────────────────────────────────────────────────────────
def fnt(size, bold=False):
    k = (size, bold)
    if k not in FONTS:
        FONTS[k] = pygame.font.SysFont("Segoe UI", size, bold=bold)
    return FONTS[k]

def txt(s, msg, size, col, pos, center=False, bold=False):
    img = fnt(size, bold).render(str(msg), True, col)
    r = img.get_rect(center=pos) if center else img.get_rect(topleft=pos)
    s.blit(img, r)
    return r

def glow(s, x, y, col, r):
    for rr in range(r, 0, -10):
        a = int(80*rr/r)
        g = pygame.Surface((rr*2,rr*2), pygame.SRCALPHA)
        pygame.draw.circle(g, (*col,a), (rr,rr), rr)
        s.blit(g, (x-rr, y-rr))

# ─── EDMONDS-KARP ME SNAPSHOT ────────────────────────────────────────────────
class EK:
    """
    Gjeneroi te gjitha hapat me snapshot-e te plota te rrjedhezave.
    flow_state[u][v] = rrjedha AKTUALE ne kete hap.
    Kjo siguron qe vizualizimi tregon saktesisht gjendjen per cdo hap.
    """
    def __init__(self):
        # Nderto strukturat e kapaciteteve dhe rrjedhezave
        self.cap  = defaultdict(lambda: defaultdict(int))
        self.adj  = defaultdict(set)
        for u,v,c in EDGES:
            self.cap[u][v] += c
            self.adj[u].add(v)
            self.adj[v].add(u)   # harku i kthyer

        # Rrjedha fillon 0 per te gjitha
        self.flow = defaultdict(lambda: defaultdict(int))

        self.steps = []   # lista e hapave per animim
        self._run()

    def _snap(self):
        """Kthe kopje te thelle te gjendjes aktuale te rrjedhezave."""
        return {u: dict(self.flow[u]) for u in self.flow}

    def _bfs(self):
        """BFS ne grafikun mbetes. Kthen (parent, harqet e eksploruara)."""
        parent   = {}
        visited  = {SOURCE}
        queue    = deque([SOURCE])
        explored = []   # (u, v) ne rendin e eksplorimit
        found    = False

        while queue and not found:
            u = queue.popleft()
            for v in self.adj[u]:
                res = self.cap[u][v] - self.flow[u][v]
                if res > 0 and v not in visited:
                    visited.add(v)
                    parent[v] = u
                    explored.append((u, v))
                    if v == SINK:
                        found = True
                        break
                    queue.append(v)
        return parent, explored, found

    def _run(self):
        iteration = 0

        while True:
            iteration += 1
            parent, explored, found = self._bfs()

            # Hapi BFS — snapshot PARA ndryshimit
            self.steps.append({
                "type":      "bfs",
                "iteration": iteration,
                "explored":  list(explored),   # lista (u,v)
                "found":     found,
                "flow_snap": self._snap(),
                "max_flow":  sum(self.flow[SOURCE][v] for v in self.adj[SOURCE]),
            })

            if not found:
                self.steps.append({
                    "type":      "done",
                    "flow_snap": self._snap(),
                    "max_flow":  sum(self.flow[SOURCE][v] for v in self.adj[SOURCE]),
                })
                break

            # Rinderto rrugen rritese
            path = []
            node = SINK
            while node != SOURCE:
                prev = parent[node]
                path.append((prev, node))
                node = prev
            path.reverse()

            # Ngushtica
            bottleneck = min(
                self.cap[u][v] - self.flow[u][v]
                for u,v in path
            )
            bottle_edge = min(
                path,
                key=lambda e: self.cap[e[0]][e[1]] - self.flow[e[0]][e[1]]
            )

            # Hapi AUGMENT — trego rrugen PARA perditesimit
            path_nodes = [SOURCE] + [v for _,v in path]
            self.steps.append({
                "type":        "augment",
                "iteration":   iteration,
                "path":        list(path),        # lista (u,v)
                "path_nodes":  path_nodes,
                "bottle_edge": bottle_edge,
                "bottleneck":  bottleneck,
                "flow_snap":   self._snap(),      # PARA ndryshimit
                "max_flow":    sum(self.flow[SOURCE][v] for v in self.adj[SOURCE]),
                "pairs": [(u,v,
                           self.cap[u][v]-self.flow[u][v],
                           self.cap[u][v])
                          for u,v in path],
            })

            # Perditeso rrjedhezat
            for u,v in path:
                self.flow[u][v] += bottleneck
                self.flow[v][u] -= bottleneck

            total = sum(self.flow[SOURCE][v] for v in self.adj[SOURCE])

            # Hapi UPDATE — snapshot PAS ndryshimit
            self.steps.append({
                "type":        "update",
                "iteration":   iteration,
                "path":        list(path),
                "bottle_edge": bottle_edge,
                "bottleneck":  bottleneck,
                "flow_snap":   self._snap(),      # PAS ndryshimit
                "max_flow":    total,
            })

# ─── VIZATIMET E NYJEVE ──────────────────────────────────────────────────────
def draw_reservoir(s, x, y, active):
    if active: glow(s, x, y, C["cyan"], 90)
    pygame.draw.circle(s, (100,120,135), (x,y), 58)
    pygame.draw.circle(s, (35, 55, 70),  (x,y), 50)
    pygame.draw.circle(s, (38,170,225),  (x,y), 42)
    pygame.draw.circle(s, (165,240,255), (x,y), 42, 3)
    for i in range(3):
        pygame.draw.arc(s, C["white"],
                        (x-34+i*8, y-20+i*8, 68, 30), 0, math.pi, 2)

def draw_pump(s, x, y, name, active, tick):
    if active: glow(s, x, y, C["green"], 70)
    pygame.draw.rect(s, (55,82,100), (x-44,y-34,88,68), border_radius=5)
    pygame.draw.rect(s, (120,158,175),(x-44,y-34,88,68), 2, border_radius=5)
    pygame.draw.rect(s, (28,42,52),  (x-15,y-4, 30,38))
    pygame.draw.circle(s, C["cyan"], (x, y-12), 15, 3)
    a0 = tick * 0.06
    for i in range(6):
        a = a0 + i*math.pi/3
        pygame.draw.line(s, C["cyan"], (x,y-12),
                         (x+int(math.cos(a)*14), y-12+int(math.sin(a)*14)), 3)
    txt(s, name, 15, C["white"], (x, y+52), center=True, bold=True)

def draw_lagje(s, x, y, name, active):
    if active: glow(s, x, y, C["purple"], 72)
    pygame.draw.rect(s, (20,32,56), (x-40,y-38,80,74), border_radius=7)
    pygame.draw.rect(s, (125,168,210),(x-40,y-38,80,74), 2, border_radius=7)
    for ox,h,col in [(-30,30,(55,88,132)),(-7,44,(48,75,118)),(17,34,(68,100,145))]:
        bx,by = x+ox, y-h//2-8
        pygame.draw.rect(s, col, (bx,by,21,h))
        pygame.draw.rect(s, (140,180,220),(bx,by,21,h), 1)
        for wy in range(by+6, by+h-4, 10):
            pygame.draw.rect(s, (255,215,100),(bx+5,wy,4,4))
            pygame.draw.rect(s, (255,215,100),(bx+13,wy,4,4))
    pygame.draw.circle(s, C["purple"], (x,y+28), 8)
    pygame.draw.circle(s, (205,185,255),(x,y+28), 8, 1)
    txt(s, name, 15, C["white"], (x, y+48), center=True, bold=True)

def draw_relief(s, x, y, active):
    if active: glow(s, x, y, C["red"], 85)
    pygame.draw.rect(s, (140,100,70), (x-43,y-43,86,86), border_radius=5)
    pygame.draw.rect(s, (225,190,145),(x-43,y-43,86,86), 2, border_radius=5)
    pygame.draw.rect(s, C["white"], (x-8, y-26,16,52))
    pygame.draw.rect(s, C["white"], (x-26,y-8, 52,16))

# ─── TUB (PIPE) ───────────────────────────────────────────────────────────────
def pipe_color(u, v, st):
    """Ngjyra e tubit bazuar ne fazën aktuale."""
    kind = st["type"]
    path  = st.get("path", [])
    bfs   = st.get("bfs_edges", [])
    be    = st.get("bottle_edge", None)

    if (u,v) == be or (v,u) == be:
        return C["red"], 9      # ngushtica — e kuqe
    if (u,v) in path or (v,u) in path:
        return C["cyan"], 8     # rruga rritese — cyan
    if (u,v) in bfs or (v,u) in bfs:
        return C["yellow"], 6   # eksploruar nga BFS — verdhe
    return C["pipe"], 5         # normal

def draw_pipe(s, x1, y1, x2, y2, flow, cap, col, width, tick, active_water):
    dx, dy = x2-x1, y2-y1
    L = math.hypot(dx,dy) or 1
    # Gjeresia proporcionale me kapacitetin
    w = max(6, width + int(cap * 0.28))

    # Shadow
    pygame.draw.line(s, C["pipe_dark"], (x1,y1),(x2,y2), w+8)
    # Tubi kryesor
    pygame.draw.line(s, col, (x1,y1),(x2,y2), w)
    # Rrjedha aktuale — brenda tubit, e kaltert
    if flow > 0 and cap > 0:
        fill_w = max(2, int(w * flow / cap))
        pygame.draw.line(s, C["water"], (x1,y1),(x2,y2), fill_w)

    # Vija e mesit
    pygame.draw.line(s, (200,230,255),(x1,y1),(x2,y2), max(1, w//6))

    # Shigjetat e drejtimit
    for t in (0.30, 0.60, 0.82):
        tx, ty = x1+dx*t, y1+dy*t
        tx2,ty2= x1+dx*(t+0.05), y1+dy*(t+0.05)
        a = math.atan2(dy,dx)
        sz = 8
        tip = (int(tx2), int(ty2))
        al  = (int(tx2-sz*math.cos(a-0.45)), int(ty2-sz*math.sin(a-0.45)))
        ar  = (int(tx2-sz*math.cos(a+0.45)), int(ty2-sz*math.sin(a+0.45)))
        c2  = C["white"] if flow > 0 else C["dim"]
        pygame.draw.polygon(s, c2, [tip,al,ar])

    # Animim uji kur aktiv
    if active_water and flow > 0:
        t_anim = (tick * 0.008) % 1
        for i in range(5):
            q = (t_anim + i/5) % 1
            wx = int(x1+dx*q); wy = int(y1+dy*q)
            pygame.draw.circle(s, (200,240,255),(wx,wy), 3)

def draw_pipe_label(s, x1, y1, x2, y2, flow, cap, col, u, v):
    """Etiketa flow/cap mbi tub me vijë lidhëse."""
    dx, dy = x2-x1, y2-y1
    L  = math.hypot(dx,dy) or 1
    nx_, ny_ = -dy/L, dx/L

    # Pozicion 45% e rrugës, offset 36px
    t  = 0.45
    bx = x1+dx*t; by = y1+dy*t
    off = 36
    # Ndrysho anen per brinjete qe mund te mbivendosen
    side = 1
    lx = int(bx + nx_*off*side)
    ly = int(by + ny_*off*side)

    # Linja lidhëse
    pygame.draw.line(s, (80,110,145),(int(bx),int(by)),(lx,ly), 1)
    pygame.draw.circle(s, (80,110,145),(int(bx),int(by)), 3)

    # Kutia e etiketës
    msg = f"{flow}/{cap}"
    lbl_w = fnt(13,True).size(msg)[0] + 16
    r = pygame.Rect(lx-lbl_w//2, ly-13, lbl_w, 25)
    pygame.draw.rect(s, (2,5,14), r, border_radius=5)
    pygame.draw.rect(s, col, r, 2, border_radius=5)

    # Ngjyra teksti — jeshil nëse ka rrjedhë, bardhë nëse jo
    tc = C["green"] if flow > 0 else (180,200,230)
    txt(s, msg, 13, tc, r.center, center=True, bold=True)

# ─── APLIKACIONI ─────────────────────────────────────────────────────────────
class App:
    def __init__(self):
        self.windowed = (1440, 860)
        self.full  = False
        self.screen= pygame.display.set_mode(self.windowed, pygame.RESIZABLE)
        pygame.display.set_caption("Edmonds-Karp — Rrjeti Urban i Ujesjellesit")
        self.clock = pygame.time.Clock()
        self.tick  = 0
        self.paused= True
        self.counter=0
        self.speed = 1   # hap per sekonde
        self.reset()

    def reset(self):
        self.ek       = EK()
        self.step_idx = 0
        self.logs     = []
        self.max_flow = 0
        self.cur      = {
            "type":       "init",
            "flow_snap":  {},
            "path":       [],
            "bfs_edges":  [],
            "bottle_edge":None,
            "bottleneck": 0,
            "path_nodes": [],
            "max_flow":   0,
            "iteration":  0,
            "found":      False,
        }
        self.log("=== Edmonds-Karp — Rrjeti i Ujesjellesit ===", C["blue"])
        self.log(f"Graf: {N_NODES} nyje, {len(EDGES)} tuba (harqe).", C["dim"])
        self.log("Te gjitha rrjedhezat fillojne nga 0.", C["dim"])
        self.log("Algoritmi gjen BFS -> Augment -> Update.", C["dim"])
        self.log("SPACE: luaj/pauzo  |  S: hap  |  R: rinicio", C["dim"])

    def log(self, msg, col):
        self.logs.append((msg,col))
        self.logs = self.logs[-22:]

    def node_pos(self):
        W, H = self.graph_w, self.screen.get_height()
        top=90; bot=H-65; span=bot-top
        return {
            0: (int(W*0.08), int(top+span*0.50)),   # Rezervuari
            1: (int(W*0.30), int(top+span*0.15)),   # SP-A
            2: (int(W*0.30), int(top+span*0.45)),   # SP-B
            3: (int(W*0.30), int(top+span*0.78)),   # SP-C
            4: (int(W*0.60), int(top+span*0.12)),   # L-1
            5: (int(W*0.60), int(top+span*0.38)),   # L-2
            6: (int(W*0.60), int(top+span*0.64)),   # L-3
            7: (int(W*0.60), int(top+span*0.88)),   # L-4
            8: (int(W*0.88), int(top+span*0.50)),   # QE
        }

    def apply_step(self):
        if self.step_idx >= len(self.ek.steps):
            return
        s = self.ek.steps[self.step_idx]
        self.step_idx += 1

        # Ndrysho gjendjen e cur duke ruajtur snapshot-in e hapi
        self.cur = {
            "type":       s["type"],
            "flow_snap":  s["flow_snap"],
            "path":       s.get("path", []),
            "bfs_edges":  s.get("explored", []),
            "bottle_edge":s.get("bottle_edge", None),
            "bottleneck": s.get("bottleneck", 0),
            "path_nodes": s.get("path_nodes", []),
            "max_flow":   s.get("max_flow", self.max_flow),
            "iteration":  s.get("iteration", self.cur.get("iteration",0)),
            "found":      s.get("found", False),
        }
        self.max_flow = s.get("max_flow", self.max_flow)

        if s["type"] == "bfs":
            it = s["iteration"]
            ne = len(s["explored"])
            if s["found"]:
                self.log(f"[{it}] BFS: gjeti rruge rritese! ({ne} harqe eksploruar)", C["yellow"])
            else:
                self.log(f"[{it}] BFS: nuk ka me rruge rritese — PERFUNDOI.", C["orange"])

        elif s["type"] == "augment":
            it  = s["iteration"]
            bn  = s["bottleneck"]
            pth = " -> ".join(NODE_NAMES[n] for n in s["path_nodes"])
            be  = s["bottle_edge"]
            self.log(f"[{it}] Rruga rritese: {pth}", C["cyan"])
            self.log(f"     Ngushtica ({NODE_NAMES[be[0]]}->{NODE_NAMES[be[1]]}): {bn} njesi", C["red"])
            for u,v,res,cap in s["pairs"]:
                self.log(f"     {NODE_NAMES[u]}->{NODE_NAMES[v]}: mbetes={res}/{cap}", C["dim"])

        elif s["type"] == "update":
            it  = s["iteration"]
            bn  = s["bottleneck"]
            tot = s["max_flow"]
            self.log(f"[{it}] Perditeso: +{bn} njesi mbi rrugen.", C["green"])
            self.log(f"     Rrjedha totale = {tot} njesi.", C["green"])

        elif s["type"] == "done":
            tot = s["max_flow"]
            self.log("=== ALGORITMI PERFUNDOI ===", C["green"])
            self.log(f"Rrjedha Maksimale = {tot} njesi/sek.", C["green"])
            self.log("Nuk ka me rruge rritese ne grafikun mbetes.", C["cyan"])
            self.log("Kjo eshte edhe kapaciteti i prerjes minimale!", C["blue"])
            self.paused = True

    def get_flow(self, u, v):
        """Merr rrjedhën e hapit aktual nga snapshot-i."""
        snap = self.cur.get("flow_snap", {})
        return max(0, snap.get(u, {}).get(v, 0))

    def draw_graph(self):
        s   = self.screen
        W,H = self.graph_w, s.get_height()
        pos = self.node_pos()
        kind= self.cur["type"]

        pygame.draw.rect(s, C["graph"], (12,80,W-24,H-128), border_radius=16)

        # Titull
        pygame.draw.rect(s, (6,10,22), (12,4,W-24,76), border_radius=10)
        txt(s,"EDMONDS-KARP — Rrjeti Urban i Ujesjellesit",
            22, C["white"], (W//2,24), center=True, bold=True)
        txt(s,"Rezervuari  ->  Stacione Pompimi  ->  Lagje  ->  Qendra Emergjente",
            13, C["blue"], (W//2,50), center=True)

        # Tuba
        path_edges = set(map(tuple, self.cur.get("path",[])))
        bfs_edges  = set(map(tuple, self.cur.get("bfs_edges",[])))
        be         = self.cur.get("bottle_edge",None)

        for u,v,cap in EDGES:
            flow = self.get_flow(u,v)
            col,width = pipe_color(u,v, self.cur)
            x1,y1 = pos[u]; x2,y2 = pos[v]
            active_water = (u,v) in path_edges
            draw_pipe(s,x1,y1,x2,y2, flow,cap, col,width, self.tick, active_water)

        # Etiketat e tubave — vizato pas tubave
        for u,v,cap in EDGES:
            flow = self.get_flow(u,v)
            col,_ = pipe_color(u,v, self.cur)
            x1,y1=pos[u]; x2,y2=pos[v]
            draw_pipe_label(s,x1,y1,x2,y2, flow,cap, col,u,v)

        # Nyjet — vizato mbi tuba
        path_nodes = set(self.cur.get("path_nodes",[]))
        for n in range(N_NODES):
            x,y = pos[n]
            active = n in path_nodes
            if n == SOURCE:
                draw_reservoir(s,x,y,active)
                txt(s,"REZERVUARI",15,C["cyan"],(x,y+68),center=True,bold=True)
                txt(s,"BURIMI",12,C["blue"],(x,y+86),center=True)
            elif n in (1,2,3):
                draw_pump(s,x,y,NODE_NAMES[n],active,self.tick)
                txt(s,"Stacion Pompimi",10,C["dim"],(x,y+70),center=True)
            elif n in (4,5,6,7):
                draw_lagje(s,x,y,NODE_NAMES[n],active)
                txt(s,"Lagjja",10,C["dim"],(x,y+64),center=True)
            else:
                draw_relief(s,x,y,active)
                txt(s,"QENDRA EMERGJENTE",14,C["red"],(x,y+72),center=True,bold=True)
                txt(s,"DERDHJA",11,C["orange"],(x,y+90),center=True)

        # Etiketa iteracioni aktual mbi nyjen k (ngjashëm me Floyd-Warshall)
        it = self.cur.get("iteration",0)
        if it > 0 and kind not in ("done","init"):
            kind_lbl = {"bfs":"BFS","augment":"AUGMENT","update":"UPDATE"}.get(kind,kind)
            r = pygame.Rect(W//2-70, H-115, 140, 28)
            col_it = {"bfs":C["yellow"],"augment":C["cyan"],"update":C["green"]}.get(kind,C["dim"])
            pygame.draw.rect(s,(8,12,28),r,border_radius=6)
            pygame.draw.rect(s,col_it,r,2,border_radius=6)
            txt(s,f"Iteracioni {it} — {kind_lbl}",13,col_it,r.center,center=True,bold=True)

        # Bara e statusit
        status = self._status()
        bar_col = {"bfs":(48,42,8),"augment":(8,48,58),"update":(8,52,18),
                   "done":(8,50,18),"init":(8,18,50)}.get(kind,(8,12,28))
        bar_brd = {"bfs":C["yellow"],"augment":C["cyan"],"update":C["green"],
                   "done":C["green"],"init":C["blue"]}.get(kind,C["dim"])
        pygame.draw.rect(s, bar_col, (12,H-46,W-24,36), border_radius=8)
        pygame.draw.rect(s, bar_brd, (12,H-46,W-24,36), 1, border_radius=8)
        txt(s, status, 13, C["white"], (W//2,H-28), center=True)

    def _status(self):
        kind = self.cur["type"]
        it   = self.cur.get("iteration",0)
        if kind == "init":   return "Inicializim — te gjitha rrjedhezat jane 0. Shtyp SPACE."
        if kind == "done":   return f"PERFUNDOI! Rrjedha Maksimale = {self.max_flow} njesi/sek"
        if kind == "bfs":
            return (f"Iter {it}: BFS gjet rruge rritese!" if self.cur["found"]
                    else f"Iter {it}: BFS — nuk ka me rruge rritese.")
        if kind == "augment":
            bn = self.cur["bottleneck"]
            be = self.cur["bottle_edge"]
            return f"Iter {it}: Rruga gjetur — ngushtica {bn} njesi te {NODE_NAMES[be[0]]}->{NODE_NAMES[be[1]]}"
        if kind == "update":
            return f"Iter {it}: Rrjedha u perditesua +{self.cur['bottleneck']} — totali = {self.max_flow}"
        return ""

    def draw_panel(self):
        s   = self.screen
        H   = s.get_height()
        px  = self.graph_w
        pw  = s.get_width() - px
        pad = 12

        pygame.draw.rect(s, (8,13,28),  (px,0,pw,H))
        pygame.draw.rect(s, (14,22,46), (px+6,6,pw-12,H-12), border_radius=12)

        kind = self.cur["type"]
        it   = self.cur.get("iteration",0)

        # ── Zona 1: Rrjedha totale — e madhe dhe e qarte ──
        txt(s,"RRJEDHA MAKSIMALE",13,C["dim"],(px+pw//2,28),center=True,bold=True)
        flow_col = C["green"] if self.max_flow>0 else C["dim"]
        txt(s,f"{self.max_flow}",52,flow_col,(px+pw//2,72),center=True,bold=True)
        txt(s,"njesi / sek",12,C["dim"],(px+pw//2,100),center=True)

        # Bar i perparimit
        total_possible = 47   # kapaciteti total teorik (burim)
        bar_fill = min(pw-pad*2, int((self.max_flow/total_possible)*(pw-pad*2)))
        bar_r = pygame.Rect(px+pad, 112, pw-pad*2, 10)
        pygame.draw.rect(s,(20,35,60),bar_r,border_radius=4)
        if bar_fill>0:
            pygame.draw.rect(s,C["green"],pygame.Rect(px+pad,112,bar_fill,10),border_radius=4)
        pygame.draw.rect(s,(40,65,100),bar_r,1,border_radius=4)

        # ── Zona 2: Info iteracioni aktual ──
        info_y = 132
        pygame.draw.line(s,(30,45,75),(px+pad,info_y),(px+pw-pad,info_y))
        info_y += 8
        kind_lbl = {"init":"GATI","bfs":"BFS","augment":"AUGMENT",
                    "update":"UPDATE","done":"PERFUNDOI"}.get(kind,kind.upper())
        col_badge = {"bfs":C["yellow"],"augment":C["cyan"],"update":C["green"],
                     "done":C["green"],"init":C["blue"]}.get(kind,C["dim"])

        badge_r = pygame.Rect(px+pad, info_y, 72, 18)
        pygame.draw.rect(s, col_badge, badge_r, border_radius=4)
        txt(s,kind_lbl,10,(4,6,16),badge_r.center,center=True,bold=True)

        if it>0:
            txt(s,f"Iter {it}",11,C["dim"],(px+pw-pad-8,info_y+2))

        info_y += 24

        # Status i detajuar
        if kind=="bfs":
            ne = len(self.cur.get("bfs_edges",[]))
            fc = C["green"] if self.cur["found"] else C["orange"]
            msg = f"Eksploroi {ne} harqe" + (" — gjeti rruge!" if self.cur["found"] else " — s'ka rruge!")
            txt(s,msg,11,fc,(px+pad,info_y))
            info_y+=16

        elif kind=="augment":
            bn  = self.cur["bottleneck"]
            be  = self.cur["bottle_edge"]
            pth = " -> ".join(NODE_NAMES[n] for n in self.cur["path_nodes"])
            # Rruga
            txt(s,"Rruga:",11,C["dim"],(px+pad,info_y)); info_y+=14
            # Shkurtojë nëse shumë e gjatë
            if fnt(11).size(pth)[0] > pw-pad*2-8:
                parts = [NODE_NAMES[n] for n in self.cur["path_nodes"]]
                for i in range(0,len(parts),2):
                    line = " -> ".join(parts[i:i+2])
                    txt(s,line,11,C["cyan"],(px+pad+8,info_y)); info_y+=14
            else:
                txt(s,pth,11,C["cyan"],(px+pad+8,info_y)); info_y+=14
            # Ngushtica
            ng_r = pygame.Rect(px+pad, info_y, pw-pad*2, 22)
            pygame.draw.rect(s,(50,10,10),ng_r,border_radius=4)
            pygame.draw.rect(s,C["red"],ng_r,1,border_radius=4)
            txt(s,f"Ngushtica: {NODE_NAMES[be[0]]}->{NODE_NAMES[be[1]]} = {bn} njesi",
                11,C["red"],ng_r.center,center=True,bold=True)
            info_y+=26

        elif kind=="update":
            bn  = self.cur["bottleneck"]
            tot = self.cur["max_flow"]
            upd_r=pygame.Rect(px+pad,info_y,pw-pad*2,22)
            pygame.draw.rect(s,(10,50,18),upd_r,border_radius=4)
            pygame.draw.rect(s,C["green"],upd_r,1,border_radius=4)
            txt(s,f"+{bn} njesi shtohen — totali: {tot}",
                11,C["green"],upd_r.center,center=True,bold=True)
            info_y+=28

        elif kind=="done":
            tot = self.cur["max_flow"]
            done_r=pygame.Rect(px+pad,info_y,pw-pad*2,44)
            pygame.draw.rect(s,(8,48,18),done_r,border_radius=6)
            pygame.draw.rect(s,C["green"],done_r,2,border_radius=6)
            txt(s,"RRJEDHA MAKSIMALE GJETUR!",12,C["green"],(px+pw//2,info_y+10),center=True,bold=True)
            txt(s,f"Max-Flow = Max-Cut = {tot} njesi",11,(180,255,180),(px+pw//2,info_y+28),center=True)
            info_y+=50

        # ── Zona 3: Gjendja e tubave aktuale ──
        flow_y = info_y + 8
        pygame.draw.line(s,(30,45,75),(px+pad,flow_y),(px+pw-pad,flow_y))
        flow_y+=6
        txt(s,"GJENDJA E TUBAVE (rrjedhe/kapacitet)",11,C["dim"],(px+pad,flow_y),bold=True)
        flow_y+=15

        snap = self.cur.get("flow_snap",{})
        path_edges = set(map(tuple,self.cur.get("path",[])))
        be_set = {tuple(self.cur["bottle_edge"])} if self.cur.get("bottle_edge") else set()

        for u,v,cap in EDGES:
            if flow_y > H-120: break
            flow = max(0, snap.get(u,{}).get(v,0))
            is_path   = (u,v) in path_edges
            is_bottle = (u,v) in be_set
            # Ngjyra
            if is_bottle: lc=C["red"]
            elif is_path: lc=C["cyan"]
            elif flow>0:  lc=C["green"]
            else:         lc=(60,75,100)

            # Bar proporcional
            bar_max = pw-pad*2-60
            bw = int(bar_max*flow/cap) if cap>0 else 0
            br = pygame.Rect(px+pad, flow_y+2, bar_max, 8)
            pygame.draw.rect(s,(15,22,40),br,border_radius=3)
            if bw>0:
                pygame.draw.rect(s,lc,pygame.Rect(px+pad,flow_y+2,bw,8),border_radius=3)

            label_str=f"{NODE_NAMES[u][:4]}->{NODE_NAMES[v][:3]} {flow}/{cap}"
            txt(s,label_str,10,lc,(px+pad+bar_max+4,flow_y-1))
            flow_y+=13

        # ── Zona 4: Log ──
        log_y = H-130
        pygame.draw.line(s,(30,45,75),(px+pad,log_y),(px+pw-pad,log_y))
        log_y+=5
        txt(s,"LOG",11,C["dim"],(px+pad,log_y),bold=True)
        txt(s,f"Hapi {self.step_idx}/{len(self.ek.steps)}",10,C["dim"],(px+pw-pad-8,log_y))
        log_y+=14

        max_c=max(26,(pw-pad*2-8)//7)
        for msg,col in self.logs[-6:]:
            log_r=pygame.Rect(px+pad,log_y,pw-pad*2,13)
            pygame.draw.rect(s,(8,12,28),log_r)
            disp=msg if len(msg)<=max_c else msg[:max_c-2]+".."
            txt(s,disp,10,col,(px+pad+3,log_y+1))
            log_y+=14
            if log_y>H-22: break

        # Controls
        for ln in ["SPACE: luaj/pauzo  |  S/->: hap  |  R: rinicio",
                   "F/F11: ekran i plote  |  ESC: dil"]:
            txt(s,ln,10,(65,85,130),(px+pad+2,log_y))
            log_y+=13

    def draw(self):
        self.screen.fill(C["bg"])
        self.draw_graph()
        self.draw_panel()
        pygame.display.flip()

    def resize(self,w,h):
        self.graph_w=int(w*0.72)

    def toggle_full(self):
        self.full=not self.full
        if self.full:
            self.screen=pygame.display.set_mode((0,0),pygame.FULLSCREEN)
        else:
            self.screen=pygame.display.set_mode(self.windowed,pygame.RESIZABLE)
        self.resize(*self.screen.get_size())

    def events(self):
        for e in pygame.event.get():
            if e.type==pygame.QUIT: return False
            if e.type==pygame.VIDEORESIZE and not self.full:
                self.windowed=(e.w,e.h)
                self.screen=pygame.display.set_mode(self.windowed,pygame.RESIZABLE)
                self.resize(e.w,e.h)
            if e.type==pygame.KEYDOWN:
                if e.key==pygame.K_ESCAPE: return False
                elif e.key==pygame.K_SPACE: self.paused=not self.paused
                elif e.key in (pygame.K_s,pygame.K_RIGHT): self.apply_step()
                elif e.key==pygame.K_r: self.reset()
                elif e.key in (pygame.K_f,pygame.K_F11): self.toggle_full()
        return True

    def run(self):
        self.resize(*self.windowed)
        running=True
        while running:
            self.tick+=1
            running=self.events()
            if not self.paused:
                self.counter+=1
                if self.counter>=max(1,60//self.speed):
                    self.apply_step()
                    self.counter=0
            self.draw()
            self.clock.tick(60)
        pygame.quit()
        sys.exit()

# ─── REZULTATET FINALE ────────────────────────────────────────────────────────
def print_results():
    ek = EK()
    # Gjej hapin e fundit
    done = next(s for s in reversed(ek.steps) if s["type"]=="done")
    snap = done["flow_snap"]
    mf   = done["max_flow"]

    print("\n"+"="*65)
    print("  EDMONDS-KARP — REZULTATET FINALE (Rrjeti i Ujesjellesit)")
    print("="*65)
    print(f"\n  Rrjedha Maksimale = {mf} njesi/sek")
    print(f"  Iteracione: {sum(1 for s in ek.steps if s['type']=='augment')}")
    print(f"  Hapa total: {len(ek.steps)}")
    print(f"  Kompleksiteti: O(V*E^2) = O({N_NODES}*{len(EDGES)}^2) = O({N_NODES*len(EDGES)**2})")

    print("\n  Rrjedhezat Finale per Tub (flow/cap):\n")
    for u,v,cap in EDGES:
        fl = max(0, snap.get(u,{}).get(v,0))
        bar= "#"*fl + "."*(cap-fl)
        pct= int(100*fl/cap) if cap else 0
        print(f"  {NODE_NAMES[u]:<12} -> {NODE_NAMES[v]:<12}: {fl:>3}/{cap:<3} [{bar}] {pct}%")

    # Bottleneck tubat (kapaciteti i shteruar)
    print("\n  Tubat e ngushticave (flow=cap, kapacitet i shteruar):")
    for u,v,cap in EDGES:
        fl=max(0,snap.get(u,{}).get(v,0))
        if fl==cap and cap>0:
            print(f"  *** {NODE_NAMES[u]} -> {NODE_NAMES[v]}: {fl}/{cap} — I MBUSHUR PLOTESISHT")

    print("\n  Perfundim:")
    print(f"  Rrjeti mund te transportoje maksimalisht {mf} njesi ujit")
    print(f"  nga Rezervuari deri ne Qendren Emergjente per sekonde.")
    print(f"  Max-Flow = Min-Cut = {mf} njesi (teorema Max-Flow Min-Cut).")
    print("="*65+"\n")

if __name__=="__main__":
    print_results()
    App().run()

pygame 2.6.1 (SDL 2.28.4, Python 3.13.5)
Hello from the pygame community. https://www.pygame.org/contribute.html

  EDMONDS-KARP — REZULTATET FINALE (Rrjeti i Ujesjellesit)

  Rrjedha Maksimale = 47 njesi/sek
  Iteracione: 8
  Hapa total: 26
  Kompleksiteti: O(V*E^2) = O(9*18^2) = O(2916)

  Rrjedhezat Finale per Tub (flow/cap):

  Rezervuari   -> SP-A        :  20/20  [####################] 100%
  Rezervuari   -> SP-B        :  15/15  [###############] 100%
  Rezervuari   -> SP-C        :  12/12  [############] 100%
  SP-A         -> L-1         :  14/14  [##############] 100%
  SP-A         -> L-2         :   6/10  [######....] 60%
  SP-B         -> L-1         :   1/8   [#.......] 12%
  SP-B         -> L-2         :   7/12  [#######.....] 58%
  SP-B         -> L-3         :   7/7   [#######] 100%
  SP-C         -> L-2         :   0/9   [.........] 0%
  SP-C         -> L-3         :   6/11  [######.....] 54%
  SP-C         -> L-4         :   6/6   [######] 100%
  L-1          -> L-2 

SystemExit: 

C:\Users\alamefl\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


---

## Lidhja Midis Floyd-Warshall dhe Edmonds-Karp

| Aspekti | Floyd-Warshall | Edmonds-Karp |
|---------|---------------|--------------|
| **Problemi** | Rrugët më të shkurtra (distancat) | Rrjedha maksimale |
| **Qasja** | Programim dinamik | BFS iterativ |
| **Output** | Matrica D[i][j] + P[i][j] | Rrjedha totale f* |
| **Kompleksiteti** | O(V³) | O(VE²) |
| **Graf** | I peshuar (me/pa drejtim) | I drejtuar me kapacitete |
| **Ideal për** | Pyetje të shumta rrugësh | Optimizim rrjetesh |
| **Shembull real** | Logjistika, navigimi urban | Ujësjellësi, interneti |

### Kur të zgjedhim çdonjërin?

**Floyd-Warshall:** Kur keni **pyetje të shumta** "cila është rruga optimale nga X në Y?"
→ Navigimi, logjistika, planifikimi i rrugëve, sisteme mjekësore urgjente

**Edmonds-Karp:** Kur doni të maksimizoni **sasinë e burimeve** që mund të transferohet
→ Ujësjellësi, rrjetet elektrike, interneti, logjistika e kapaciteteve

### A mund të kombinohen?

Po! Një shembull i mundshëm:
1. Përdor **Floyd-Warshall** për të gjetur rrugët minimale midis të gjitha stacioneve
2. Ndërto **rrjetin e kapaciteteve** ku pesha e çdo harku është 1/D[i][j]
3. Zbato **Edmonds-Karp** mbi rrjetin e ri për maksimum-flow

Kjo qasje hibride përdoret në probleme si shpërndarja optimale e burimeve mjekësore.

---

## Pyetje Reflektuese

1. **Floyd-Warshall:** Pse duhet të provojmë çdo kulm si ndërmjetës, jo vetëm fqinjët?
2. **Floyd-Warshall:** Çfarë ndodh nëse grafi ka cikle negative dhe si e detekton algoritmi?
3. **Floyd-Warshall:** Pse O(V³) është e pranueshme për rrjetin logjistik (V=8) por jo për GPS-in e Europës (V=10⁶)?
4. **Edmonds-Karp:** Pse BFS (jo DFS) garanton kompleksitetin O(VE²)?
5. **Ushtrimi 2:** Pse SA-2 doli si qendra më e aksesueshme? Si ndikon topologjia e grafit?
6. **Lidhja:** A mund të përdorni Floyd-Warshall brenda Edmonds-Karp? Si dhe pse?
7. **Real:** Rrjeti i internetit: cilën strukturë ka (flow network apo shortest path) dhe pse?

---

*Laboratori 8 — Algoritme mbi Grafe | Floyd-Warshall & Edmonds-Karp*
